# Bigtable

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/bigtable.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Bigtable?

Bigtable is a distributed, highly scalable, and NoSQL database developed by Google.

**Mental model.** Bigtable is a sparse, distributed, sorted map. A table maps one row key to one row, and a row holds column families, each with columns (qualifiers) and versioned cells. The map is sorted, so a range of row keys is one cheap scan.

**How it works.** Data is split into tablets (contiguous key ranges) served by a cluster of nodes. Nodes hold no durable data: SSTables live in Colossus, Google's distributed file system, so storage and compute scale and fail independently. This is the key difference from a single-node store.

**Gotcha.** Bigtable is not SQL and has no joins. If a query needs a join, an ad hoc `GROUP BY`, or a full-table aggregation, it belongs in BigQuery, not here.

Worked example: one table, one row key, four dimensions.

```text
row key:        acme#dev-00042#2026-10-04T12:00:00Z
column family:  r
column:         temperature
cell:           value=21.4, timestamp=2026-10-04T12:00:00Z
```

The same row key can carry `humidity`, `battery`, and later versions of `temperature`. Sparse means rows do not need the same columns, so the schema never has to be padded with nulls.

Common mistake: treating a table as relational and adding one row per column. Model one entity per row and put the varying fields in columns.

## How do you design a good Bigtable row key?

A good Bigtable row key supports the query pattern and avoids hotspotting. Bigtable stores rows lexicographically by key, so monotonically increasing keys like raw timestamps can concentrate writes on one tablet. Use prefixes, salting, bucketing, or reversed timestamps when needed.

Example:

```text
customer_id#reverse_timestamp
region#bucket#device_id#timestamp
```

In [ ]:
from datetime import datetime

MAX_TS = 9_999_999_999   # far-future sentinel for reversed timestamps

def row_key(tenant: str, device_id: str, ts: datetime) -> str:
    # salt spreads hot tenants across tablets; reversed timestamp sorts newest first
    bucket = hash((tenant, device_id)) % 16
    reverse_ts = MAX_TS - int(ts.timestamp())
    return f"{bucket:02d}#{tenant}#{device_id}#{reverse_ts:010d}"

The same idea with lexicographic prefixes:

```text
00#acme#dev-00042#9999999999   -> newest first inside one device
00#acme#dev-00042#9999999900
01#globex#dev-00007#9999999998
```

**Mental model.** The row key is the only index, so it must encode the access pattern. Rows sort lexicographically by raw bytes. A prefix scan reads every row whose key starts with a prefix, in sorted order, from one contiguous key range.

**How it works.** Put the field you filter on first, then the field you sort by, then a unique suffix. Keep keys short (tens of bytes); long keys cost storage and cache on every row. Use fixed-width, zero-padded numbers so byte order matches numeric order: `2026-10-04` sorts correctly, `10/4/2026` does not.

**Gotcha.** A key that starts with a monotonically increasing value (a timestamp, a sequence, a `user-00001` counter) sends all new writes to one tablet end. That is the classic hotspot.

Worked example: queries drive the key.

```text
Query: latest readings for one device, newest first.
Key:   acme#dev-00042#9999999999   (reversed timestamp for newest-first)
Query: all devices in a region over a time window.
Key:   region#device_id#ts         (region first, so a region is one scan)
```

Common mistake: choosing the key from the entity shape instead of the query list. Write down the top three reads and writes first, then design the key to make each a prefix scan or a point lookup.

## When should you use Bigtable instead of BigQuery?

Use Bigtable for low-latency operational lookups, time-series access, and high-throughput key-value style workloads. Use BigQuery for analytical SQL, joins, scans, and aggregations. Bigtable is not a general data warehouse; it is optimized for serving rows by key/range.

Example:

+ Bigtable: fetch recent IoT readings for one device in milliseconds
+ BigQuery: calculate monthly usage by region across all devices

Many architectures use both: Bigtable for serving and BigQuery for analytics.

**Mental model.** Bigtable serves individual rows and narrow key ranges at low latency and high request rates. BigQuery scans columns across billions of rows for analytics. They answer different questions: "what is this user's state now" versus "what did all users do last quarter".

**How it works.** A Bigtable read is an index seek plus a few disk or cache reads, usually single-digit milliseconds. A BigQuery query reads columnar files in parallel and is billed by bytes scanned, so it prefers large scans and dislikes point lookups.

**Gotcha.** Running analytics on Bigtable forces full-table scans through a serving path and crowds out application traffic. Keep them apart, or read analytics through BigQuery federation or Data Boost.

Worked example: the same event stream, two serving needs.

```text
Serving:  get latest 20 events for user u-7781        -> Bigtable point + range read
Analytics: p95 latency by region for the last 90 days -> BigQuery scan
```

Common mistake: putting a hot dashboard on Bigtable "because the data is already there". Dashboards scan many rows and belong on a columnar copy.

Many architectures use both: Bigtable for serving, BigQuery for analytics, Change Streams or Dataflow to keep the copy fresh.

## How does Bigtable handle data storage?

Bigtable stores data in a sparse, distributed, and multi-dimensional sorted map.

**Mental model.** Bigtable stores a sparse, distributed, multidimensional sorted map. Each cell is `(row key, column family, column qualifier, timestamp) -> bytes`. Unset cells cost nothing, which is why wide and sparse rows are normal.

**How it works.** A write lands in a tablet node's commit log and in-memory memtable. When the memtable fills it flushes to an immutable SSTable in Colossus. A compaction later merges SSTables and drops versions the GC policy rejects. Nodes serve from cache, memtable, and SSTables; durable bytes always live in Colossus.

**Gotcha.** Storage is decoupled from nodes, so adding nodes does not add storage in the same way it would on a local-disk system. Size the cluster for throughput, and let storage grow independently.

Worked example: the write path in order.

```text
1. client sends mutation to the tablet node that owns the key range
2. node appends to the commit log (durability) and the memtable (readable now)
3. memtable flush -> SSTable in Colossus
4. compaction merges SSTables and applies the GC policy
```

Common mistake: assuming a delete frees space immediately. A delete writes a tombstone, and the bytes go away only when compaction removes the tombstoned versions.

## What are the key features of Bigtable?

Some key features of Bigtable include scalability, high performance, fault tolerance, and automatic load balancing.

**Mental model.** The important features are the ones that change how you design: a sorted row-key map, automatic sharding and rebalancing, linear horizontal scale, single-row atomicity, per-family garbage collection, replication across clusters, and IAM plus CMEK security.

**How it works.** You create an instance, one or more clusters, and tables. Bigtable splits and moves tablets for you, so there are no shards to place and no rebalancing jobs to run. GC runs automatically at compaction using the rules you set per column family.

**Gotcha.** "Automatic" does not mean "no design". The row key and the GC policy are still yours, and they determine whether the cluster is balanced and affordable.

Worked example: what you configure versus what the service does.

```text
You configure:  row key, column families, GC policy, app profiles, node count
Bigtable does:  tablet split and merge, rebalancing, compaction, GC, failover
```

Common mistake: expecting a knob for every HBase setting. Bloom filters, block cache sizing, and region pre-splitting are service-managed here, not user settings.

## How does Bigtable achieve scalability?

Bigtable achieves scalability by partitioning data into tablets, which are distributed across multiple servers.

**Mental model.** A table is many tablets, each a contiguous key range. Tablets are distributed across the cluster's nodes. Adding nodes spreads tablets further, so read and write capacity grows roughly linearly with node count.

**How it works.** The service splits a tablet when it grows, and it can merge small adjacent tablets. Because data lives in Colossus and nodes are stateless, a tablet can be moved to another node without copying durable data. Hotspots are the one thing scaling cannot fix: if all traffic hits one key range, extra nodes sit idle.

**Gotcha.** Scale is per row-key range, not per table. A single hot row has a hard ceiling no matter how many nodes you buy.

Worked example: capacity follows key spread.

```text
8 nodes, evenly spread keys  -> each node serves about 1/8 of the tablets
8 nodes, all writes keyed by today's date -> one tablet end takes the load
```

Common mistake: scaling out to fix latency caused by a hot key prefix. Diagnose the prefix distribution first; salting or field promotion is the fix.

## What is a tablet in Bigtable?

A tablet is a range of rows in a Bigtable that is stored and managed independently by a single server.

**Mental model.** A tablet is a contiguous range of row keys and the unit of distribution and load balancing, roughly the region of HBase. A table is a set of non-overlapping tablets that together cover the whole key space.

**How it works.** The service splits a tablet as it grows and merges neighbours when they shrink, so the number of tablets tracks data size rather than a fixed shard count. Tablets move between nodes as load changes, and clients are told which node serves which range.

**Gotcha.** The number of tablets is not something you set. You cannot pre-split or manually split a tablet in Cloud Bigtable, so a key that creates one huge range is fixed only by changing the key.

Worked example: key range to tablets.

```text
acme#dev-00001 .. acme#dev-09999  -> tablet A
acme#dev-10000 .. acme#dev-24999  -> tablet B
globex#...                        -> tablet C
```

Common mistake: assuming one tablet per table or one per node. The service decides, and it rebalances as the data grows.

## How does Bigtable ensure high performance?

Bigtable achieves high performance by leveraging in-memory and distributed storage, as well as employing efficient indexing techniques.

**Mental model.** Performance comes from three things: a key layout that spreads load, a working set that fits cache, and client behaviour that batches and limits reads. The service handles the rest with automatic sharding and compaction.

**How it works.** Point lookups touch the memtable or cache and return in single-digit milliseconds. Range scans stream rows in key order. Node throughput has a documented ceiling, so Google recommends planning for roughly 10,000 simple reads or writes per second per node and keeping sustained utilization well below the limit.

**Gotcha.** One wide request can be slower than many small ones. Reading 10,000 columns of a wide row blocks that row's tablet and costs more than a filtered read of the few columns you need.

Worked example: targeting capacity.

```text
Peak 50,000 reads/s and 20,000 writes/s
Plan at about 70 percent of the node ceiling:
  read nodes  = ceil(50,000 / (10,000 * 0.7)) = 8
  write nodes = ceil(20,000 / (10,000 * 0.7)) = 3
Use 8 nodes, because reads and writes share the same nodes.
```

Common mistake: sizing on average traffic and ignoring the daily peak. Size on peak, then let autoscaling follow the curve.

## How does Bigtable ensure fault tolerance?

Bigtable maintains multiple replicas of each tablet, ensuring data durability and availability in case of server failures.

**Mental model.** Durable data lives in Colossus with replication, so a dead node loses nothing. Compute is stateless, so the service reassigns its tablets to healthy nodes. A multi-cluster instance adds another copy in a second zone or region.

**How it works.** The commit log and SSTables are replicated by the storage layer. Bigtable detects an unhealthy node, moves its tablets, and clients retry against the new location. Replication between clusters is asynchronous and eventual.

**Gotcha.** Fault tolerance is not backup. A bad application write or a dropped table replicates to every cluster in milliseconds; only a backup restores it.

Worked example: two failure types, two protections.

```text
node or zone failure -> automatic, service-managed (replication + tablet reassignment)
bad deploy, dropped table -> restore from a backup into a new table
```

Common mistake: treating multi-cluster replication as a restore point. It copies mistakes, not history.

## How does Bigtable handle load balancing?

Bigtable automatically redistributes tablets across servers to balance the workload and maintain performance.

**Mental model.** The service moves tablets between nodes so no node serves too many hot ranges. Balancing reacts to load signals, not just to byte counts, so a hot tablet is spread even if the table's data is uneven.

**How it works.** As tablets split, the new tablets are placed on different nodes. Nodes that are overloaded offload tablets; underloaded nodes pick them up. Replication and locality constraints (for example keeping data near an app profile's cluster) bound where a tablet may go.

**Gotcha.** Rebalancing cannot split a single hot row. If every request uses the same row key, all of it stays on one tablet on one node.

Worked example: balancing versus key design.

```text
even key spread + hot node   -> service rebalances and fixes it
one hot row key              -> nothing to rebalance; change the key
```

Common mistake: watching node CPU and adding nodes when the real problem is one hot row. Check the per-tablet and per-prefix metrics first.

## What consistency model does Bigtable provide?

Within a single cluster, Bigtable provides strong consistency: a read sees the result of every completed write. When an instance has clusters in more than one region, replication between them is asynchronous, so data converges across clusters instead of being immediately consistent. App profile routing decides which behavior a client gets: single-cluster routing for strict consistency, multi-cluster routing for lower-latency replicated reads.

**Mental model.** Within a single cluster, reads are strongly consistent: after a write commits, every later read sees it. Across clusters, replication is asynchronous, so a read from another cluster may briefly see an older version.

**How it works.** A normal app profile uses single-cluster routing and reads and writes the same cluster, so consistency is immediate. A multi-cluster app profile can route reads anywhere for lower latency, and accepts eventual consistency in exchange.

**Gotcha.** Do not mix the two expectations. If the application needs read-your-writes, pin it to one cluster, or read the row back from the cluster that accepted the write.

Worked example: choosing routing by need.

```text
read-your-writes required      -> single-cluster app profile
nearest-region reads required  -> multi-cluster app profile, eventual consistency
```

Common mistake: enabling multi-cluster routing to cut latency, then filing a bug that a just-written value is missing. That is the documented trade-off, not a fault.

## How does Bigtable support structured data?

Bigtable stores data as byte arrays, allowing developers to interpret the data in any structured format they desire.

**Mental model.** Structure comes from naming, not from a declared schema. A column family groups related columns, and a column qualifier is an arbitrary byte string. You can encode `address#city` and `address#zip` as separate qualifiers under one family.

**How it works.** Qualifiers sort lexicographically inside a family, so a shared prefix gives you a range read of related fields. You can also store a serialized `bytes` payload in one cell, or use a protobuf schema with Bigtable's schema bundles for typed access.

**Gotcha.** There are no types enforced by the server; everything is bytes. The client or a schema encodes the type, so two writers can disagree about the format of the same qualifier.

Worked example: qualifier layout for one entity.

```text
family: profile
  address#city   -> "Austin"
  address#zip    -> "78701"
  name           -> "Ada"
family: events
  2026-10-04#click -> "1"
```

Common mistake: putting the entity id inside the qualifier. The row key already identifies the entity; repeating it wastes bytes on every cell.

## What are some typical use cases for Bigtable?

Bigtable is commonly used for storing and analyzing large amounts of time-series data, as well as for serving real-time applications.

**Mental model.** The fit is a large number of entities, each read or written by key or by a narrow ordered range, at high request rates with low and predictable latency. If the access is a scan and aggregate, it is a warehouse job.

**How it works.** Common shapes are time-series telemetry, per-user state and message history, feature serving, real-time bidding profiles, personalization, graph or index stores, and recommendation inputs. All share a key-first access pattern.

**Gotcha.** Bigtable is a poor fit for multi-row transactions, ad hoc joins, and full-text relevance ranking. Use a relational store, a warehouse, or a search engine for those.

Worked example: matching query to system.

```text
latest reading for a device         -> Bigtable
sum of usage by region for a month  -> BigQuery
full-text search over messages      -> a search engine, fed from Bigtable
```

Common mistake: forcing every access pattern into one store. Most production systems pair Bigtable with a warehouse and a search index.

## How does Bigtable handle schema changes?

Bigtable is schema-less, meaning that new columns can be added to the table without affecting existing rows.

**Mental model.** There is almost no schema to migrate. Adding a column family is an online admin operation, and adding a new column qualifier is just writing it. Existing rows simply do not have the new cell, which is what sparse means.

**How it works.** `cbt createfamily` or the client library adds a family without downtime. Removing a family drops its data. New qualifiers appear the first time a writer uses them, and readers must tolerate a missing cell.

**Gotcha.** Changing a GC policy is not retroactive in the way people expect; it takes effect as data is compacted, so old versions can linger briefly after you tighten the rule.

Worked example: adding a field with no migration.

```bash
cbt -project acme-prod -instance acme-bt createfamily device_readings battery
# writers start setting r:battery; older rows return no cell for it
```

Common mistake: writing migration jobs to backfill a new column. Read code should treat a missing cell as a default, and backfill only if analytics needs it.

## Can you explain the architecture of Bigtable?

Bigtable has a distributed architecture with multiple components, including tablets, tablet servers, and a master server for coordination.

**Mental model.** Four moving parts: clients, a metadata layer that maps key ranges to nodes, the tablet nodes that serve reads and writes, and Colossus that stores the SSTables and commit logs. Data and compute are separated.

**How it works.** A client asks which node owns a key, caches the answer, and sends the request directly to that node. The node reads cache, memtable, or SSTables and returns rows. The service splits, merges, and moves tablets in the background.

**Gotcha.** The row key is the shard key. Architecture diagrams often show even sharding, but uneven key prefixes produce uneven tablets regardless of how many nodes exist.

Worked example: request path for a point read.

```text
client -> metadata lookup (cached) -> tablet node for the key range
       -> memtable/cache/SSTable -> row bytes -> client
```

Common mistake: drawing Bigtable as a single primary with replicas for scale. There is no single primary for data; nodes are peers serving different key ranges.

## How does Bigtable handle data replication?

Bigtable replicates data across multiple data centers to ensure durability and availability in case of failures.

**Mental model.** Replication is per instance and per cluster. You add a second cluster in another zone or region, and Bigtable copies writes to it asynchronously. Each cluster holds a full copy of the table.

**How it works.** A write goes to one cluster first and is then replicated to the others. An app profile decides where a client reads and writes: one cluster for strong consistency, or any cluster for low-latency reads with eventual consistency. You can also route by cluster group to keep some data regional.

**Gotcha.** Replication is not a transaction across clusters. There is no cross-cluster atomic commit, and conflicting concurrent writes to the same cell resolve by last-write-wins on the server timestamp.

Worked example: two clusters, two app profiles.

```text
instance acme-bt
  cluster acme-bt-us-east1   (us-east1)
  cluster acme-bt-eu-west1   (europe-west1)
app profile "serving"  -> multi-cluster routing, reads nearest
app profile "billing"  -> single-cluster acme-bt-us-east1, read-your-writes
```

Common mistake: assuming a failover promotes a primary. There is no primary; clients reroute through the app profile.

## What are the advantages of using Bigtable over traditional relational databases?

Some advantages of Bigtable include scalability, high performance, fault tolerance, and the ability to handle unstructured data efficiently.

**Mental model.** Bigtable trades query flexibility and multi-row transactions for horizontal scale, predictable key-based latency, and hands-off operation. It stores billions of rows without sharding code and scales by adding nodes.

**How it works.** There is no fixed schema, no joins, and no secondary indexes, so writes are simple log plus memtable appends and reads are single index seeks. The service handles splitting, rebalancing, compaction, and replication.

**Gotcha.** You give up SQL, joins, and cross-row ACID. A workload that needs those will reimplement them in the application, usually badly, so pick the store that matches the workload.

Worked example: the trade in one line each.

```text
Relational: flexible queries, transactions, vertical scale limits
Bigtable:   fixed key access, single-row atomicity, horizontal scale
```

Common mistake: choosing Bigtable for a workload whose first requirement is a multi-table transaction. That requirement does not get easier at scale.

## Does Bigtable support ACID transactions?

Bigtable gives ACID semantics at the row level: a row mutation, including a conditional read-modify-write, is atomic. Cross-row transactions are not the design center, so model around single-row atomicity instead of expecting relational-style multi-row transactions.

**Mental model.** Bigtable supports atomicity at the row level, not across rows. A single row mutation, which may touch many columns, applies as one unit. Conditional mutations give you compare-and-set on a row.

**How it works.** Use `CheckAndMutateRow` (Python: `row.commit()` after a conditional row, or the `conditional_row` API) to apply changes only if a predicate holds. This supports read-modify-write and optimistic concurrency within one row.

**Gotcha.** A batch that spans rows is not atomic. If one mutation in a batched write fails, others may already have applied, so retry logic must be idempotent.

Worked example: atomic counter update on one row.

```python
row = table.conditional_row(b"acme#counter")
row.set_cell("c", b"value", b"1")
if_true = row_filters.ValueRegexFilter(b"^0$")
row.commit()  # applies only when the predicate matches
```

Common mistake: moving money between two rows in one operation. Do it as two idempotent single-row mutations with a reconciliation step, or keep that data in a transactional database.

## How can you interact with Bigtable?

Bigtable provides client libraries for different programming languages, such as Java, Python, and Go, to interact with the database.

**Mental model.** Four client paths: the `cbt` command line for admin and debugging, the language client libraries for applications, the GoogleSQL surface for row-key queries, and BigQuery federation for analytics. All talk to the same service.

**How it works.** `cbt` is best for creating tables, families, and GC policies and for reading a few rows. The Python, Java, Go, and other clients give connection pooling, batching, and retries. The HBase client library works against Bigtable for HBase migrations.

**Gotcha.** `cbt` is not a production driver. It opens a connection per command and has no pooling, so use it for operations and use a client library for application traffic.

Worked example: the everyday commands.

```bash
cbt -project acme-prod -instance acme-bt ls
cbt -project acme-prod -instance acme-bt ls device_readings
cbt -project acme-prod -instance acme-bt read device_readings prefix=acme#
cbt -project acme-prod -instance acme-bt count device_readings
```

Common mistake: scripting `cbt` in a hot path. It is a debugging and admin tool, not a low-latency client.

## What is the recommended way to perform atomic row-level updates in Bigtable?

Bigtable supports atomic row-level updates using conditional mutations, allowing you to update multiple cells within a row atomically.

**Mental model.** Treat a row as the unit of consistency. Read the row, decide, and write it back with a conditional mutation so a concurrent writer cannot interleave. If the condition fails, re-read and retry.

**How it works.** `CheckAndMutateRow` takes a predicate filter and two mutation sets, one for true and one for false, and applies exactly one of them atomically. The read and the write are not a transaction, so you use the condition to detect interference and retry.

**Gotcha.** An unconditional read-modify-write is a lost-update bug under concurrency. Always guard it with a predicate, or use an aggregate family for counters.

Worked example: optimistic concurrency with a version cell.

```python
row = table.conditional_row(b"acct-7781")
row.set_cell("s", b"balance", new_balance)
row.set_cell("s", b"version", str(version + 1).encode())
row.set_cell("s", b"pending", b"1", state=False)  # clear only on success
row.commit()
```

Common mistake: reading outside the conditional and assuming the value is still current. Put the expected version in the predicate, not in application memory alone.

## How does Bigtable handle access control and security?

Bigtable integrates with Google Cloud IAM (Identity and Access Management) to provide fine-grained access control and security policies.

**Mental model.** Security has three layers: IAM roles for who may call which API, encryption for data at rest and in transit, and audit logging for who did what. There is no row-level SQL grant; access is per table through IAM.

**How it works.** Roles like `roles/bigtable.reader`, `roles/bigtable.user`, and `roles/bigtable.admin` are granted at the project, instance, or table level. Data is encrypted at rest by default, optionally with CMEK, and in transit with TLS. Data Access audit logs record reads and writes but are off by default.

**Gotcha.** IAM is not row-level. If two tenants must not see each other's rows, use separate tables or instances, or authorized views, not a column filter in the client.

Worked example: least privilege for a service account.

```bash
gcloud bigtable instances add-iam-policy-binding acme-bt \
  --member="serviceAccount:reader@acme.iam.gserviceaccount.com" \
  --role="roles/bigtable.reader"
```

Common mistake: granting project-wide `bigtable.admin` to an application so it can read one table. Grant the narrowest role at the narrowest resource.

## Can you describe the data model used in Bigtable?

Bigtable uses a sparse, distributed, and multidimensional sorted map, where data is indexed by a row key, column key, and timestamp.

**Mental model.** A table is a sorted map: row key to row. A row contains column families, a family contains columns (qualifiers), and a column contains cells, each a version stamped with a timestamp. The full address of a value is the tuple `(row, family, qualifier, timestamp)`.

**How it works.** Rows sort by row key bytes. Qualifiers sort within a family. Versions sort newest timestamp first. Timestamps are 64-bit values, usually microseconds; the server sets them by default, or you supply your own for import and replay.

**Gotcha.** The row key is the only automatic index. Reading by any other field means a full scan or a second table that you maintain as a secondary index.

Worked example: one address, three versions.

```text
table:         device_readings
row key:       acme#dev-00042#2026-10-04T12:00:00Z
family:        r
qualifier:     temperature
cells:         (ts=12:00:00, "21.4") (ts=11:59:00, "21.3") (ts=11:58:00, "21.1")
```

Common mistake: assuming a family is a column. A family is a container with one GC policy; the columns inside it are created on write and are not declared.

In [ ]:
from google.cloud import bigtable

# review-only: requires the google-cloud-bigtable package and a real instance
client = bigtable.Client(project="acme-prod", admin=True)
table = client.instance("acme-bt").table("device_readings")

row = table.direct_row(b"acme#dev-00042#2026-10-04T12:00:00Z")
row.set_cell("r", b"temperature", b"21.4", timestamp_micros=1_759_579_200_000_000)
row.set_cell("r", b"humidity", b"48")
row.commit()


## How does Big table handle data sharding and distribution?

Bigtable automatically shards data by range partitioning the row keys and distributes tablets across multiple servers.

**Mental model.** Sharding is by row-key range, not by a hash ring or a fixed partition count. The service cuts the key space into tablets and distributes those tablets; the boundaries are data-driven and move as data grows.

**How it works.** A tablet splits when it grows, and the two halves can land on different nodes. Adjacent small tablets merge. Because data lives in Colossus, moving a tablet is a metadata change, not a bulk copy.

**Gotcha.** Distribution is only as even as the key distribution. Sequential keys push every new row to one end of the key space, so one tablet takes all the writes while the rest idle.

Worked example: uneven versus even keys.

```text
key = 2026-10-04T12:00:00Z#dev-42   -> all writes hit the last tablet
key = 07#dev-42#9999999999          -> writes spread over 16 salt buckets
```

Common mistake: hashing the whole key and losing range scans. Salt only the leading portion so the rest still sorts.

## What is the maximum size of a row in Bigtable?

Bigtable enforces hard system limits on row key size, row size, and cell size (see the Cloud Bigtable quotas and limits page), and oversized rows hurt latency and write amplification. Design many small rows rather than a few huge ones, and keep large blobs in Cloud Storage with a reference in Bigtable.

**Mental model.** A row key is limited to 4 KiB, and the practical limit on a row is much smaller than a relational row. Bigtable is designed for many small rows, not a few huge ones.

**How it works.** A single cell value has a service maximum on the order of 100 MiB, but that ceiling is for rare blobs, not for routine data. Wide rows with thousands of columns can be written, but reading them is slow and they pin the tablet serving that key.

**Gotcha.** A row that grows without bound, such as one row per user holding all their events, becomes a hotspot and a slow read. Split time-ordered data across many rows.

Worked example: capping the row.

```text
bad:  row key = user-7781, qualifier = event#evt-0001  (unbounded row, one per user)
good: row key = user-7781#9999999999, qualifier = event  (one row per event)
```

Common mistake: storing a list in one cell and appending to it. Each append rewrites the whole value and creates a version; store one event per row instead.

## How does Bigtable handle data compression?

Bigtable provides built-in support for data compression, allowing you to save storage space and improve read and write performance.

**Mental model.** Bigtable compresses data automatically. Compression is not a table or column-family setting you choose, and the service picks algorithms per workload.

**How it works.** SSTables are compressed in Colossus, which cuts storage cost and the bytes read per seek. Similar values next to each other in key order compress better, so a well-clustered key layout also improves the compression ratio.

**Gotcha.** Client-visible payloads are not compressed by the service, so a huge cell still travels over the wire at its raw size. Compress large blobs in the application before storing them.

Worked example: what compression does and does not cover.

```text
storage in Colossus   -> compressed automatically
read bytes per seek   -> reduced by compression
wire payload to client-> not compressed by Bigtable
```

Common mistake: assuming Snappy is a knob to set. On Cloud Bigtable you do not pick the algorithm; on HBase you do, which is one of the differences between the two.

## Does Bigtable support secondary indexes?

There is no classic relational secondary index, and the traditional answer was to duplicate data under a second row-key pattern. Current releases add asynchronous secondary indexes and continuous materialized views, plus GoogleSQL for Bigtable, so less of that duplication has to be hand-built.

**Mental model.** There is no built-in secondary index. The row key is the only ordered index. Any other lookup path is a second table that you keep in sync, or a BigQuery copy.

**How it works.** The standard pattern is an index table whose rows are `indexed_value#primary_key` and whose cells point back to the main row. Writes update both rows, ideally with Change Streams and Dataflow rather than a fragile dual write. Continuous materialized views and asynchronous secondary indexes are also available for some patterns.

**Gotcha.** An index table must handle the same hotspot rules as any table. Indexing a low-cardinality field, like `status`, puts millions of rows under a few prefixes.

Worked example: index by email.

```text
main table:  row = u-7781            family profile: email = "ada@acme.com"
index table: row = ada@acme.com#u-7781   family i: pk = "u-7781"
lookup:      read index row, then read main row
```

Common mistake: writing the index inline with the main write and ignoring partial failure. Use Change Streams or a conditional reservation so the two stay consistent.

## Can you explain the role of a tablet server in Bigtable?

A tablet server in Bigtable hosts and serves a set of tablets, handling read and write requests for the data within those tablets.

**Mental model.** Tablet nodes are stateless compute. They serve reads and writes for the tablets assigned to them, hold the memtable and cache, and stream SSTables from Colossus. In Cloud Bigtable you never manage one; you manage nodes per cluster.

**How it works.** A node handles requests for its tablets, buffers writes, flushes to SSTables, and runs compaction. If a node fails, its tablets are reassigned and the durable data is already in Colossus, so no data is lost.

**Gotcha.** Because nodes hold no durable data, "one node" is a throughput unit, not a storage unit. Adding nodes raises QPS, not the size of the table you can store.

Worked example: per-node accounting.

```text
nodes:        3
each node:    serves some tablets, holds cache and memtable
durable data: all SSTables in Colossus, independent of node count
```

Common mistake: sizing nodes by table size. Size nodes by requests per second and keep storage growth separate.

## How does Bigtable handle hotspots?

Splitting helps, but a monotonically increasing row key still funnels writes into one tablet until it splits, so salting or bucketing the key prefix is the real fix. Watch per-tablet load and CPU utilization in Cloud Monitoring, not just table-level throughput.

**Mental model.** A hotspot is a row key, or a small key range, that receives far more traffic than the rest. It happens for two reasons: sequential keys (timestamps, counters) concentrate writes at one end, and a single popular entity concentrates all reads on one row.

**How it works.** Because rows sort by key, appending `2026-10-04T12:00:00Z` sends every new row to the highest key. A famous user or a global config row creates a single-tablet read hotspot. Key Visualizer shows the hot ranges as bright bands over time.

**Gotcha.** Adding nodes does not fix a hotspot; the hot range still lives on one tablet. Fix the key: promote a high-cardinality field to the front, add a hash or salt prefix, or reverse the timestamp.

Worked example: two fixes.

```text
write hotspot: 2026-10-04T12:00:00Z#device-42
  fix:         42#device-42#9999999999   (salt prefix spreads writes)
read hotspot:  CONFIG#global-flags
  fix:         01#CONFIG#global-flags  (16 buckets) + a cache in the app
```

Common mistake: salting the middle of the key. The salt must be the first bytes, or it does not change the sort order or the tablet boundaries.

## Does Bigtable support full-text search capabilities?

Bigtable is not a search engine. For full-text search over this data, export or stream it to Elasticsearch/OpenSearch or Vertex AI Search. Newer Bigtable releases add k-nearest-neighbor vector search through GoogleSQL for Bigtable, which covers similarity search rather than classic text indexing.

**Mental model.** Bigtable does not do relevance ranking or tokenized search. It stores and returns bytes. If you need ranked search over text, Bigtable is the source of truth and a search engine is the index.

**How it works.** The usual pipeline writes documents to Bigtable and streams the same changes through Change Streams or Dataflow into a search engine such as Elasticsearch or a vector index. Bigtable answers "give me document d-42"; the search engine answers "which documents mention latency".

**Gotcha.** Prefix matching on a row key is not text search. `prefix=lat` matches keys that start with those bytes, not words anywhere in a value.

Worked example: split the roles.

```text
Bigtable:     row = doc-42, family doc: body = "..."
search index: tokenized copy of body, built from Change Streams
query:        search engine returns doc ids -> point reads from Bigtable
```

Common mistake: scanning a text column in Bigtable to find a substring. That is a full-table scan and it will not rank results.

## How does Bigtable handle backups and disaster recovery?

Bigtable provides built-in backup and restore functionality, allowing you to create backups and restore data to a specific point in time.

**Mental model.** A backup is a consistent, independent copy of a table that you create on demand and restore into a new table. Replication copies current state; a backup preserves a point in time that replication has already overwritten.

**How it works.** A backup is stored in a cluster and carries an optional expiration; by default it does not expire, and you delete it yourself. Restore creates a new table, then you point the application at it. You can also copy a backup to another region for cross-region recovery.

**Gotcha.** A backup is not continuous. Between two backups, a bad write or a dropped table loses everything since the last one, so schedule backups against your recovery point objective.

Worked example: create and restore.

```bash
gcloud bigtable backups create acme-daily-2026-10-04 \
  --instance=acme-bt --cluster=acme-bt-us-east1 --table=device_readings
gcloud bigtable tables restore device_readings_restored \
  --instance=acme-bt --cluster=acme-bt-us-east1 \
  --source=projects/acme-prod/instances/acme-bt/clusters/acme-bt-us-east1/backups/acme-daily-2026-10-04
```

Common mistake: restoring over the live table. Restore always targets a new table name; cut over deliberately.

## Can you explain the difference between Bigtable and HBase?

Bigtable and HBase are similar in many ways, as HBase was inspired by Bigtable. The main difference lies in their underlying infrastructure: Bigtable runs on Google's infrastructure, while HBase runs on top of the Hadoop ecosystem.

**Mental model.** Bigtable and HBase share a design lineage: the Bigtable paper inspired HBase, and both use a sorted map, tablets or regions, memtables, and SSTables or HFiles. Bigtable is the managed Google service; HBase is the open-source system you run yourself.

**How it works.** HBase runs a Master, RegionServers, ZooKeeper, and HDFS, and you tune block cache, Bloom filters, region pre-splits, and compaction. Bigtable runs on Colossus with stateless nodes, manages splitting and compaction for you, and exposes IAM, replication, and autoscaling as service features.

**Gotcha.** The APIs differ. HBase has coprocessors, filters, and a richer client API that Bigtable does not fully support; Bigtable has no ZooKeeper or HDFS to operate, which is the main reason teams move.

Worked example: operational surface.

```text
HBase:    you run Master, RegionServers, ZooKeeper, HDFS; you pre-split regions
Bigtable: you run no servers; you set nodes, app profiles, and GC policies
```

Common mistake: migrating an HBase key design with sequential timestamps unchanged. The hotspot behaviour is the same, so a schema review is part of the migration.

## What are the considerations for choosing between Bigtable and other databases like Cassandra or MongoDB?

The choice depends on factors such as data volume, query patterns, scalability requirements, and the need for tight integration with other Google Cloud services.

**Mental model.** Ask three questions: what is the access pattern, what consistency does the application require, and who operates the system. Cassandra and MongoDB are self-managed (or managed on another cloud); Bigtable is a managed Google service with a key-range model.

**How it works.** Bigtable and Cassandra both scale horizontally with a wide-column model, but Cassandra uses consistent hashing and tunable quorum consistency, while Bigtable uses sorted key ranges and single-cluster strong reads. MongoDB is a document store with secondary indexes and richer queries, which Bigtable lacks.

**Gotcha.** Choosing on features alone misses operations. If the team cannot run a Cassandra ring or a MongoDB replica set, a managed service changes the total cost more than any benchmark.

Worked example: fit by requirement.

```text
ordered range scans by key, managed      -> Bigtable
multi-cloud, tunable quorum, self-run    -> Cassandra
ad hoc queries and secondary indexes     -> MongoDB (or a relational store)
```

Common mistake: picking Bigtable for a document workload that needs many secondary indexes. Build those indexes explicitly or choose the document store.

## How does Bigtable handle compaction?

Bigtable performs compaction by periodically merging smaller sorted files into larger ones, reducing storage overhead and improving read performance.

**Mental model.** Compaction merges immutable SSTables into fewer, larger ones and applies the garbage collection policy in the same pass. It is background work that reclaims space and keeps reads fast by limiting the number of files a read must check.

**How it works.** A flush produces a new SSTable from the memtable. Minor compactions merge small SSTables; major compactions rewrite a larger set and drop expired or tombstoned versions. Because GC happens during compaction, a tightened policy takes effect gradually.

**Gotcha.** Compaction consumes node CPU and disk bandwidth. A burst of writes produces many flushes, and the resulting compaction work competes with serving traffic, so size nodes for headroom, not for the average.

Worked example: from memtable to compacted SSTables.

```text
write -> memtable -> flush -> sstable-1
write -> memtable -> flush -> sstable-2
                 compaction merges sstable-1 + sstable-2, applies GC -> sstable-3
```

Common mistake: expecting a delete to reduce storage immediately. The tombstone and the old value remain until compaction merges the files that hold them.

## Can you explain how Bigtable manages garbage collection?

Bigtable uses an automatic garbage collection process to reclaim disk space by removing older versions of data that are no longer needed.

**Mental model.** GC is per column family and rule-based. A cell is kept or dropped by the rule when compaction processes it. There are two primitives, `max_num_versions` and `max_age`, and two combinators, `union` and `intersection`.

**How it works.** `max_num_versions` keeps the newest N versions of each cell. `max_age` keeps cells newer than a duration. `union` keeps a cell if any rule keeps it; `intersection` keeps it only if every rule keeps it. You set the rule with `cbt setgcpolicy` or the client library, and `cbt ls device_readings` shows the current policy.

**Gotcha.** GC runs at compaction, not at write time, so an expired cell can still be read until compaction reaches it. Read code must filter by timestamp if it needs strict freshness.

Worked example: the rules as JSON.

```text
keep newest 3 versions:      {"max_num_versions": 3}
keep 30 days:                {"max_age": "2592000s"}
newest 3 OR 30 days:         {"union": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
newest 3 AND 30 days:        {"intersection": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
```

The same policies from the command line:

```bash
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxversions=3
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxage=30d
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxversions=3 or maxage=30d
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxversions=3 and maxage=30d
cbt -project acme-prod -instance acme-bt ls device_readings
```

Common mistake: using `union` when the goal is to cap storage. Union is the permissive rule; it keeps a cell if either rule keeps it. Use `intersection` to keep only cells that satisfy both.

## How does Bigtable handle data locality?

Bigtable provides a mechanism called Colossus locality, which optimizes data placement to minimize network traffic and improve performance.

**Mental model.** Locality has two meanings. Within a cluster, rows near each other in key order are stored together, so a prefix scan reads few tablets. Across regions, replication places a full copy of the data in another cluster, closer to users.

**How it works.** The key prefix determines which tablet holds a row, so co-locating related rows is a key-design decision. For geographic locality, you add a cluster in the target region and route reads through a multi-cluster app profile.

**Gotcha.** There is no way to pin a specific row to a specific node. You influence locality through the key and through cluster and app-profile routing, nothing finer.

Worked example: locality by design.

```text
key = region#device#ts   -> one region's data is contiguous, scans are local
replica cluster in europe-west1 -> european reads served near users
```

Common mistake: expecting a colocation hint like a relational placement directive. The key layout is the only fine-grained lever.

## What is the role of the Bigtable master server?

The Bigtable master server handles administrative tasks such as tablet assignment, load balancing, and metadata management.

**Mental model.** Cloud Bigtable has no master server that you run or that owns the data. The metadata layer tracks which node serves which key range; the data plane is a peer set of stateless tablet nodes.

**How it works.** The service assigns tablets to nodes, splits and merges them, and updates client routing. HBase's Master and ZooKeeper have no direct equivalent you operate. In the admin API, instances, clusters, tables, and app profiles are the resources you manage.

**Gotcha.** Interview answers that describe a Bigtable master as a write bottleneck or a single point of failure for data are wrong. The control plane manages metadata; it does not proxy row traffic.

Worked example: who does what.

```text
control plane: table and cluster metadata, tablet placement, splits
data plane:    stateless nodes serving reads and writes for assigned tablets
your job:      keys, families, GC policy, node count, app profiles
```

Common mistake: looking for a `master` in the Bigtable API. The closest concepts are the instance and cluster resources.

## How does Bigtable handle concurrent read and write requests?

Bigtable uses optimistic concurrency control, where multiple readers and writers can access the same row simultaneously, ensuring consistency during conflicts.

**Mental model.** Reads and writes do not block each other. A write appends to the commit log and memtable; a read sees the memtable plus SSTables. Multiple versions coexist, and a read returns the newest version by default.

**How it works.** Requests to different rows proceed in parallel on different tablets. Requests to the same row are ordered by the serving node. Concurrency control between a read and a later write is the application's job, using conditional mutations.

**Gotcha.** A read-then-write without a condition can lose an update. Two clients can read the same value, both write, and the second write silently wins.

Worked example: lost update.

```text
t1: A reads counter = 10
t2: B reads counter = 10
t3: A writes 11
t4: B writes 11   <- A's increment is lost
```

Common mistake: assuming Bigtable serializes your read-modify-write. Wrap it in a conditional mutation and retry on predicate failure.

## Does Bigtable provide automatic indexing for faster querying?

Every row is indexed by its row key automatically, and that is the index the storage engine relies on for point lookups and range scans. For access patterns the row key cannot express, use asynchronous secondary indexes, continuous materialized views, or a duplicated row-key layout.

**Mental model.** No. There is one index, the row key. Every other query path is either a range scan of that index or a table you maintain as a secondary index.

**How it works.** Range scans on a row-key prefix are fast because they use the index order. Lookups by a non-key field scan rows, which is why the standard answer is an index table, Change Streams, or a BigQuery copy. Bigtable also offers continuous materialized views and asynchronous secondary indexes as managed building blocks.

**Gotcha.** An index table doubles write volume and inherits the hotspot rules. Indexing a low-cardinality field concentrates rows under a few prefixes.

Worked example: index table layout.

```text
main:   row = u-7781
index:  row = ada@acme.com#u-7781   -> points to u-7781
read:   index lookup, then main lookup (two point reads)
```

Common mistake: calling a prefix scan an index. It only works when the queried field is the leading part of the key.

## How does Bigtable handle time-based data, such as event logs?

Bigtable uses a timestamp associated with each cell, allowing you to store and query time-series data efficiently.

**Mental model.** Time is part of the key, not only a cell timestamp. Put the entity first and a reversed timestamp second so the newest events sort first and a prefix scan returns the latest N without reading history.

**How it works.** A key like `device-42#9999999999` gives newest-first ordering for one device. A GC policy of `max_age` then drops old events automatically, so storage stays bounded without a cleanup job. For queries across devices, put the grouping field first: `2026-10-04#device-42#ts`.

**Gotcha.** A raw ascending timestamp as the leading key field is the classic hotspot: every new event goes to the last tablet. Reverse it, salt it, or bucket by hour.

Worked example: latest-first scan.

```text
key = device-42#9999999990, device-42#9999999890, ...
scan prefix=device-42#, limit 10 -> newest ten rows, one range read
GC policy max_age=7d on the family -> older events expire at compaction
```

Common mistake: storing the timestamp only as the cell timestamp and keying by event id. Then "latest for this device" needs a full scan of the device's rows.

## Can you explain the role of a Bloom filter in Bigtable?

A Bloom filter is a probabilistic data structure used by Bigtable to reduce disk I/O by filtering out irrelevant data during read operations.

**Mental model.** A Bloom filter answers "this file definitely does not contain the key" cheaply, so reads skip SSTables that cannot hold the row. In HBase you choose filter types and tune them; in Cloud Bigtable the service manages this internally.

**How it works.** When a read looks for a row, Bigtable consults per-file metadata before touching the file. A negative answer skips a disk read, which matters when many SSTables cover a key range. The service decides the filter scope from the workload.

**Gotcha.** Bloom filters help point reads, not scans. A scan of many rows must read the data anyway, so a filter cannot make it cheaper.

Worked example: where the filter saves work.

```text
point read of device-42
  sstable-1: filter says no   -> skip
  sstable-2: filter says maybe -> read and find the row
```

Common mistake: hunting for a Bigtable Bloom filter setting. Unlike HBase, there is no `BLOOMFILTER` column-family attribute to set here.

## What is the role of the Bigtable client library?

The Bigtable client library provides the necessary APIs and interfaces to interact with Bigtable, making it easier to read, write, and manipulate data.

**Mental model.** The client library turns application calls into gRPC requests with routing, retries, batching, and connection pooling. It also caches which node owns which key range so requests go directly to the right tablet node.

**How it works.** Python, Java, Go, and other libraries expose admin APIs (create table, family, GC policy, app profile) and data APIs (read row, read rows, mutate row, conditional row, bulk writes). The HBase client library speaks the HBase API for migrations.

**Gotcha.** Batch size and retry settings are application concerns. A batch that is too large hurts tail latency, and a retry without idempotency can double-apply a mutation.

Worked example: the Python entry points.

```python
from google.cloud import bigtable
client = bigtable.Client(project="acme-prod", admin=True)
instance = client.instance("acme-bt")
table = instance.table("device_readings")
table.mutate_rows(rows)   # batched writes
table.read_rows(start_key=b"acme#", end_key=b"acme$")
```

Common mistake: creating a client per request. Create it once and reuse it so connection pooling works.

## Does Bigtable support automatic scaling of storage and compute resources?

Yes, Bigtable can automatically scale storage and compute resources based on workload patterns and configuration settings.

**Main point.** Storage scales automatically with the data. Compute scales when you change the node count, and autoscaling can do that for you within a minimum and maximum.

**How it works.** SSDs and Colossus grow with the table with no disk to provision. Autoscaling watches CPU utilization and request rate and adds or removes nodes between your bounds, with a target utilization. Storage usage and node count are separate concerns and separate costs.

**Gotcha.** Autoscaling cannot fix a hotspot. If the load is concentrated on one key range, adding nodes does not spread it, and the cluster stays hot and expensive.

Worked example: an autoscaling policy.

```text
min nodes: 3   max nodes: 12
target CPU utilization: 60%
storage: grows with the table, independent of node count
```

Common mistake: setting the minimum to one node for a production table and relying on scale-up to absorb a traffic spike. Scaling is not instant, so budget headroom.

## Can you explain the concept of tablet splitting in Bigtable?

Tablet splitting is the process of dividing a tablet into two or more smaller tablets to evenly distribute the data and workload across servers.

**Mental model.** Splitting cuts a tablet into two contiguous key ranges so the data can be served by two nodes. It is automatic and driven by size and load; you neither request nor prevent it.

**How it works.** When a tablet grows past the service threshold, it splits near the middle of its key range by byte size. The halves may move to different nodes, and adjacent small tablets can merge later. This is why a table has no fixed shard count.

**Gotcha.** A split cannot separate rows that share the same key prefix if that prefix is itself hot. If one prefix gets most traffic, all its tablets can still be busy while others idle.

Worked example: splitting a growing tablet.

```text
before: tablet covers acme#dev-00001 .. acme#dev-99999
after:  tablet A covers acme#dev-00001 .. acme#dev-49999
        tablet B covers acme#dev-50000 .. acme#dev-99999
```

Common mistake: assuming a split divides traffic evenly. It divides the key range, and traffic follows the keys.

## How does Bigtable handle time-travel queries?

Bigtable allows you to retrieve previous versions of data by specifying a timestamp or a time range in your queries.

**Mental model.** There is no `AS OF` query and no system time-travel. Versioned cells let you read older values if they are still present, bounded by the column family's GC policy.

**How it works.** Each cell keeps multiple versions until compaction drops them. A read can ask for all versions or filter by a timestamp range, so you can reconstruct recent history. Backup and restore covers point-in-time recovery at table granularity.

**Gotcha.** History is not guaranteed. Once `max_num_versions` or `max_age` has dropped a version, it is gone. Never rely on a cell version as an audit log; write changes to a dedicated table or stream them out.

Worked example: reading older versions.

```python
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"device-42#",
    end_key=b"device-42$",
    filter_=row_filters.CellsColumnLimitFilter(5),  # newest five versions
)
```

Common mistake: promising "we can query any past state" from cell versions. Say instead that history is bounded by the GC policy, and use backups for recovery.

## Does Bigtable support change data capture (CDC) for real-time data integration?

Bigtable supports change data capture with Change Streams: you enable a change stream on a table and read mutation records from it, usually with Dataflow. Managed templates stream changes to BigQuery or Pub/Sub, which is the modern path for keeping a warehouse, cache, or downstream service in sync with Bigtable.

**Mental model.** Change Streams is Bigtable's CDC feature. You enable it on a table, and Bigtable appends a record of every row change to a changelog that a stream processor reads within seconds.

**How it works.** A change stream can capture the latest value or all values. A Dataflow or Beam job reads the stream and writes it to BigQuery, Pub/Sub, or another sink. The retention window for streamed records is configurable and capped at a few days, so consumers must keep up.

**Gotcha.** A change stream is not a full history and not a transaction log. Records can be delivered more than once, so the sink must be idempotent.

Worked example: serving plus analytics.

```text
device_write -> Bigtable tables/device_readings (change stream enabled)
             -> Dataflow reads change stream -> BigQuery analytics
```

Common mistake: dual-writing to Bigtable and BigQuery from the application and hoping they stay consistent. Let the change stream be the single path.

## Can you explain the role of Bigtable's compression algorithm, Snappy?

Snappy is a fast and efficient compression algorithm used by Bigtable to reduce the size of stored data and improve read and write performance.

**Mental model.** Snappy is an HBase-era detail. In an HBase cluster you choose a compression codec per column family, and SNAPPY is a common low-CPU choice. Cloud Bigtable compresses storage automatically and does not expose a codec setting.

**How it works.** The Bigtable wire protocol and storage layer handle compression internally. Your lever over the compression ratio is data layout: values that are similar and adjacent in key order compress better. Short, well-ordered keys and small cells compress well.

**Gotcha.** Do not answer a Cloud Bigtable interview question by describing how to enable Snappy. The correct point is that compression is managed and that key design affects the ratio.

Worked example: what you can influence.

```text
not a knob: the codec, block size, or compression policy
a lever:    key order and cell size, which change compressibility
```

Common mistake: quoting HBase `COMPRESSION => 'SNAPPY'` as Cloud Bigtable syntax. It is real HBase DDL, not a Bigtable setting.

## How does Bigtable handle data replication across regions?

Bigtable uses cross-region replication to asynchronously replicate data to multiple regions, ensuring data durability and availability in case of regional failures.

**Main point.** You add a cluster in another region and Bigtable replicates the tables to it asynchronously. An app profile then decides whether clients read the nearest copy or a single cluster.

**How it works.** Each cluster stores a full copy. Writes land first in one cluster and propagate; a multi-cluster app profile can serve reads from any cluster, while a single-cluster profile keeps read-your-writes. You can also define cluster groups so specific app profiles route only to selected clusters.

**Gotcha.** Cross-region replication adds cost (another cluster and egress) and a consistency window. It lowers read latency for distant users; it does not make a single global write atomic.

Worked example: regional routing.

```bash
cbt -project acme-prod -instance acme-bt listclusters
cbt -project acme-prod -instance acme-bt createcluster acme-bt-eu \
  --zone=europe-west1-b --num-nodes=3
```

Common mistake: adding a replica cluster and expecting writes to be faster. Writes still go to one cluster; replication is asynchronous traffic behind them.

## What is the impact of schema design on Bigtable performance?

```text
few column families, separated by access pattern:
  cf_hot    small, read on every request
  cf_events append-only, expired by GC
  cf_blob   rarely read, large values that could move to GCS instead
```


**Mental model.** Schema design is key design plus family design. The row key sets distribution and query cost; column families set GC behaviour and how much data a read touches.

**How it works.** A leading high-cardinality field spreads load. A reversed timestamp makes newest-first reads one scan. Few families avoid extra lookups per read, since a scan across N families touches N family streams. Small cells keep reads light.

**Gotcha.** One giant family with hundreds of qualifiers makes every read scan the row's whole column space for the family unless you filter. Split hot and cold data into separate families so GC and reads treat them differently.

Worked example: family split by access and lifetime.

```text
family profile: small, rarely written, no GC expiry
family events:  append-heavy, max_age=30d
read path:      read only the profile family on the hot request
```

Common mistake: putting everything in one default family because "there is no schema". Families are the GC and read-scope boundary, so the choice matters.

## Does Bigtable support integration with popular data processing frameworks like Apache Spark or Apache Beam?

Yes, Bigtable integrates with popular data processing frameworks like Apache Spark and Apache Beam, allowing seamless data processing and analysis.

**Main point.** Yes. There is a Bigtable Spark connector and a Beam connector, plus Dataflow templates and a Kafka sink connector. These are the standard paths for batch export, import, and streaming.

**How it works.** The Spark connector reads and writes Bigtable tables so batch jobs can export to Parquet or BigQuery. The Beam and Dataflow connectors handle bounded and unbounded pipelines, including Change Streams to BigQuery. Import and export also run through Dataflow templates.

**Gotcha.** Bulk scans compete with serving traffic. Use Data Boost or a replicated cluster for large read jobs so analytics does not slow the application path.

Worked example: where each tool fits.

```text
Spark:    batch export of a table to Parquet or BigQuery
Beam/Dataflow: streaming Change Streams to BigQuery or Pub/Sub
Data Boost: serverless reads for large jobs, isolated from serving
```

Common mistake: running a full-table Spark scan against the live serving cluster at peak. Schedule it, or use a read-isolated path.

## Can you explain how Bigtable handles data versioning?

Bigtable assigns a unique timestamp to each cell, allowing multiple versions of a cell's data to be stored and retrieved.

**Mental model.** Every cell holds a list of timestamped versions, newest first. A write adds a version; it never overwrites history in place. A read returns the newest version unless you ask for more.

**How it works.** The server assigns the timestamp when you do not supply one, in microseconds. You can write your own timestamp for imports, replay, or out-of-order data. GC policy controls how many versions survive, and compaction enforces it.

**Gotcha.** A custom timestamp must respect the GC rules. If you import with old timestamps and the family uses `max_age`, the imported data can be garbage-collected almost immediately.

Worked example: version list on one cell.

```text
family r, qualifier temperature:
  ts 2026-10-04T12:00:00Z  "21.4"   <- read returns this
  ts 2026-10-04T11:59:00Z  "21.3"
  ts 2026-10-04T11:58:00Z  "21.1"
```

Common mistake: using a non-monotonic custom timestamp and then wondering why the newest value does not appear. Reads sort by timestamp descending.

## What is the role of Bigtable's memstore?

Bigtable's memstore is an in-memory data structure that temporarily holds recently written data before flushing it to disk.

**Mental model.** The memtable (memstore in HBase terms) is the in-memory write buffer for a tablet. Writes become readable as soon as they are in the memtable and the commit log, without waiting for a disk flush.

**How it works.** Each tablet node buffers writes for its tablets, then flushes to an SSTable when the buffer fills. A read checks the memtable first, then cache, then SSTables. The service sizes and manages the memtable; there is no user knob.

**Gotcha.** A write burst triggers many flushes, and the later compactions compete with serving traffic for CPU. This is why write-heavy tables need node headroom, not just storage.

Worked example: read path order.

```text
read row -> memtable (newest writes) -> block cache -> SSTable(s) in Colossus
```

Common mistake: quoting HBase `hbase.hregion.memstore.flush.size` as a Cloud Bigtable setting. It is a real HBase tuning parameter with no Bigtable equivalent.

## How does Bigtable handle schema evolution?

Bigtable is schemaless at the row level: rows can carry different columns, and new column qualifiers appear without a migration. Removing a column means deleting its cells or letting garbage collection expire them. Column families are the fixed part, and changes to them are online operations.

Mental model: scheme evolution is additive and lazy. New qualifiers need no migration; new families are an online admin call; removing a family deletes its data.

**How it works.** Because rows are sparse and cells are untyped bytes, a reader must treat a missing cell as absent or default. A second writer can change the value encoding without a coordinated migration.

**Gotcha.** There is no registry enforcing types. Two services writing `r:status` as `"ACTIVE"` and as `b"1"` produce a table that one of them cannot read. Document the encoding or use schema bundles.

Worked example: safe additive change.

```text
v1 writers: profile: email
v2 writers: profile: email, profile: email_verified
v1 readers: ignore email_verified (they never request it)
```

Common mistake: repurposing an existing qualifier with a new meaning. Old rows keep the old format, so readers cannot tell the two apart.

## Does Bigtable provide automatic data expiration?

In [ ]:
from google.cloud.bigtable import column_family

# keep the newest 3 versions and drop anything older than 30 days
rule = column_family.UnionRule(
    column_family.MaxVersionsGCRule(3),
    column_family.MaxAgeGCAgeRule(30 * 24 * 60 * 60),
)
table.column_family("r", gc_rule=rule).create()

```bash
# inspect and change a family's GC policy from the CLI
cbt -project acme-prod -instance acme-bt ls device_readings
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxage=7d
```

**Main point.** Yes, through GC policy, not through a TTL column. A `max_age` rule on a column family expires cells after a duration, and compaction removes them.

**How it works.** Set `max_age` in seconds with the client library, or `maxage=30d` with cbt. You can combine it with `max_num_versions` using union or intersection. Expiration is per family, so hot data with a short life and reference data with no expiry can share a table.

**Gotcha.** Expiration is not immediate. A cell past its age is removed when compaction processes it, so reads may still see it briefly. Use a read-time timestamp filter when freshness must be exact.

Worked example: TTL on an events family.

```bash
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings r maxage=7d
cbt -project acme-prod -instance acme-bt ls device_readings
```

Common mistake: storing the TTL as a value and writing a nightly delete job. Let the GC policy do the work; it needs no scanner and no downtime.

## Can you explain how Bigtable handles data encryption?

Bigtable encrypts data at rest with Google-managed keys by default and supports customer-managed encryption keys (CMEK) through Cloud KMS. Data in transit is protected with TLS. Encrypting individual sensitive fields is an application-level concern.

**Mental model.** Data is encrypted at rest by default and in transit with TLS. You can add CMEK to control the key, and you can enable audit logging to record data access.

**How it works.** At-rest encryption uses Google-managed keys unless you configure a Cloud KMS key for CMEK on the instance or cluster. TLS protects client connections. IAM decides who can call which API.

**Gotcha.** Encryption protects bytes, not access. An application with `bigtable.reader` reads every row in the table, so tenant isolation still needs separate tables, instances, or authorized views.

Worked example: enabling CMEK.

```bash
gcloud bigtable instances update acme-bt --instance=acme-bt \
  --encryption-kms-key=projects/acme-prod/locations/us-east1/keyRings/bt/cryptoKeys/bt-key
```

Common mistake: confusing CMEK with customer-supplied keys or client-side encryption. CMEK still lets the service read the data.

## How does Bigtable handle data access from different regions?

Bigtable routes read and write requests to the closest replica within a region, minimizing network latency for data access.

Mental model: replicate the data to a cluster near the users, then route reads there with an app profile. Writes still originate in one cluster and replicate asynchronously.

**How it works.** A multi-cluster app profile routes requests to the nearest healthy cluster. Single-cluster routing pins a client to one cluster for read-your-writes. Cluster groups let you keep some clients limited to specific clusters.

**Gotcha.** Latency to the nearest replica is only half the story. A write from a distant client still crosses to the cluster that accepts it, so write-heavy global apps may prefer a regional write path plus aggregation.

Worked example: routing by client.

```text
app profile "eu-serving": multi-cluster, group = [acme-bt-eu-west1, acme-bt-us-east1]
app profile "us-ledger":  single-cluster acme-bt-us-east1
```

Common mistake: assuming one app profile fits all services. Serving and billing usually need different routing and consistency.

## Does Bigtable provide support for aggregations and analytics?

Bigtable is optimized for key-based reads and writes, not for scanning aggregates. In order of preference: maintain aggregates at write time (counter families or aggregate cells), use continuous materialized views, query with GoogleSQL for Bigtable, use Data Boost for large read jobs, or query the data from BigQuery. Hadoop or Dataflow is the heavy-lift path, not the first answer.

**Mental model.** Bigtable serves key lookups; it is not an analytics engine. Two managed paths bridge the gap: aggregate column families for write-time counters, and BigQuery federation for analytical reads.

**How it works.** Aggregate families maintain sum, min, max, and similar values as writes arrive, so you read a running total instead of scanning. BigQuery can query a Bigtable table through an external (federated) table, and Data Boost serves large reads without touching serving capacity.

**Gotcha.** Aggregates only cover the values you configured, and federation is for row-key-shaped reads, not arbitrary joins. Real analytics still belongs on a columnar copy.

Worked example: BigQuery over Bigtable.

```sql
CREATE EXTERNAL TABLE analytics.device_readings
OPTIONS (
  format = 'CLOUD_BIGTABLE',
  uris = ['https://googleapis.com/bigtable/projects/acme-prod/instances/acme-bt/tables/device_readings'],
  bigtable_options = '{"columnFamilies":[{"familyId":"r","columns":[{"qualifier":"temperature","type":"BYTES","encoding":"TEXT"}]}]}'
);

SELECT rowkey, temperature
FROM analytics.device_readings
WHERE rowkey LIKE 'acme#dev-00042%'
LIMIT 100;
```

Common mistake: running `GROUP BY` over a full federated table without a row-key filter. It reads everything and is slow and expensive; export to BigQuery instead.

## Can you explain how Bigtable handles write amplification?

Bigtable minimizes write amplification by buffering and batching smaller writes into larger, more efficient ones before flushing them to disk.

**Mental model.** Write amplification is the extra bytes written for each logical byte: commit log, memtable flush, and repeated compaction rewrites. Bigtable hides the machinery but the cost is real in node CPU and background I/O.

**How it works.** Heavy rewrites of the same row increase versions, so compaction must merge and drop them. Wide rows that append one column at a time create a new version of the row region repeatedly. A GC policy with no version cap grows the version count and the compaction work.

**Gotcha.** Constantly updating one row, such as a counter or a status field, concentrates both the hotspot and the write amplification on that key. Use an aggregate family or spread counters across buckets.

Worked example: counter contention.

```text
bad:  one row "global-counter", every event increments it -> hotspot + versions
good: 16 rows "counter#00" .. "counter#15", sum on read -> parallel writes
```

Common mistake: ignoring versions when tuning writes. Cap `max_num_versions` so compaction has less to merge.

## How does Bigtable handle data locality in a multi-region setup?

Bigtable replicates data across regions, allowing read and write requests to be served from the closest replica, reducing network latency.

**Mental model.** Each cluster holds a full copy in its region, so reads from that region are local. Routing is controlled by the app profile, and replication keeps copies current with a small lag.

**How it works.** A multi-cluster profile sends a client to the nearest cluster. Cluster groups restrict a profile to a subset of clusters. Replication is asynchronous, so a local read can be momentarily stale.

**Gotcha.** You do not choose which rows replicate where; replication is per cluster and covers the tables in the instance. Region-pinned subsets need separate instances or cluster groups with separate app profiles.

Worked example: placing clusters.

```text
clusters: us-east1, europe-west1, asia-southeast1
app profile "global-read": multi-cluster -> nearest healthy cluster
```

Common mistake: expecting a read to prefer a replica with the freshest data. Routing is by proximity and health, not by freshness.

## Can you explain how Big table handles data access control on a per-row basis?

IAM grants are at instance and table level, not per row. For row-level restriction, use authorized views (which scope an application to a row-key prefix and a column set) or isolate tenants in separate tables.

**Mental model.** Bigtable does not filter rows by IAM. IAM controls access to a table or instance, not to a key range inside it. Per-row control is an application concern.

**How it works.** Options include separate tables or instances per tenant, authorized views that expose a subset of a table, and application-side filtering after a read. Authorized views are the managed way to share a slice of a table without copying it.

**Gotcha.** Reading a row and then deciding not to return it is not access control; the client already holds the bytes. Put the isolation boundary in the storage layout or in the service tier.

Worked example: tenant isolation choices.

```text
strongest: instance or table per tenant, IAM at that resource
managed:   authorized view exposing only the tenant's rows
weakest:   one table, application filters after reading
```

Common mistake: telling an auditor that a `WHERE tenant = 'acme'` clause in the client enforces row-level security.

## Does Bigtable support automatic data partitioning?

Yes. Bigtable automatically partitions rows into tablets and splits, merges, and rebalances them as data grows or hot spots appear. The part you design is the row key, so that writes spread across tablets instead of concentrating on one range.

**Main point.** Yes. Tablets split and merge automatically as data grows and shrinks, so there is no partition count to choose and no partition maintenance.

**How it works.** The service splits a tablet when it grows and reassigns the pieces to spread load. Because storage is in Colossus, repartitioning is a metadata change. Client libraries cache the key-range map and refresh it as boundaries move.

**Gotcha.** Automatic partitioning follows key distribution. Sequential keys create a moving hot end that the service cannot split away from, since the new keys all land in the same tablet.

Worked example: what causes a split.

```text
tablet grows in bytes -> service splits at a key boundary -> two tablets, maybe two nodes
tablet shrinks        -> adjacent small tablets can merge
```

Common mistake: trying to pre-split or force a split. There is no user API for that in Cloud Bigtable.

## Can you explain the concept of Bigtable's compaction and memtable?

Compaction is the process of merging smaller sorted files into larger ones to improve storage efficiency. Memtable is an in-memory buffer for recent writes before compaction.

**Mental model.** The memtable is the mutable in-memory buffer; SSTables are immutable sorted files. Compaction is the background merge that turns many SSTables into fewer and applies GC.

**How it works.** A write appends to the commit log and memtable. A full memtable flushes to a new SSTable. Compaction merges SSTables, keeps the newest version per cell, drops versions the GC policy rejects, and removes tombstones. Reads merge the memtable with the SSTables they need.

**Gotcha.** Compaction is the only place GC happens, so storage does not shrink the moment a version expires. Plan capacity for the interval between flushes and the next compaction.

Worked example: the lifecycle.

```text
memtable -> flush -> sstable-1
memtable -> flush -> sstable-2
compaction(sstable-1, sstable-2) -> sstable-3  (GC applied here)
```

Common mistake: thinking the memtable is a cache. It holds unflushed writes and is part of durability via the commit log.

## How does Bigtable handle concurrent access to the same row?

Mutations to a single row are atomic, so concurrent read-modify-write operations cannot partially interleave. Bigtable does not expose client-visible row locks: concurrent writes to the same cell are resolved by timestamp, and clients implement optimistic concurrency with conditional mutations (for example, compare-and-set on a version column).

**Mental model.** Writes to one row are atomic and applied in order by the serving node. Readers see the memtable plus SSTables, so they never see half a mutation. Two writers still race at the application level.

**How it works.** A single `MutateRow` touching several columns of one row is atomic. `CheckAndMutateRow` adds a condition so only one of two racing writers succeeds. Different rows are served independently and do not contend.

**Gotcha.** Last-write-wins on the same cell is silent. If two writers set the same cell without a condition, the later timestamp wins and the earlier value is a retained version, not an error.

Worked example: compare-and-set.

```python
from google.cloud.bigtable import row_filters

row = table.conditional_row(b"device-42#9999999999")
row.set_cell("r", b"state", b"claimed")
row.set_cell("r", b"owner", b"worker-3", state=False)
row.commit()
```

Common mistake: using a distributed lock service for a counter that one conditional mutation could handle.

## Can you explain how Bigtable handles garbage collection of older versions of data?

Bigtable periodically identifies and removes older versions of data during the compaction process to reclaim disk space.

**Mental model.** GC evaluates each cell against the family rule during compaction. Older versions drop when the rule no longer keeps them, and tombstoned data drops after its tombstone has covered all older copies.

**How it works.** `max_num_versions` keeps the newest N. `max_age` drops cells older than the duration. Union keeps if any rule keeps; intersection keeps only if all keep. Applied per family, per cell, at compaction time.

**Gotcha.** A delete of a single cell writes a tombstone; the data is gone logically at once but physically only after compaction. Storage metrics lag the delete for this reason.

Worked example: intersection versus union.

```text
intersection(max_num_versions=3, max_age=30d): keep only if BOTH hold
union(max_num_versions=3, max_age=30d):        keep if EITHER holds
```

Common mistake: assuming the default policy drops old versions aggressively. Check the family policy before you rely on version history.

## Does Bigtable support data replication within a single region?

Yes, Bigtable supports data replication within a single region to provide higher availability and durability.

**Main point.** Yes. Clusters are zonal, and two clusters can sit in two zones of one region for in-region high availability and lower-latency failover. Zone-level replication is the common baseline.

**How it works.** An instance can hold clusters in different zones, and Bigtable replicates tables across them. An app profile with multi-cluster routing fails over to the healthy zone automatically. This protects against a single-zone outage without cross-region latency.

**Gotcha.** Two zones in one region do not protect against a whole-region failure. Add a cross-region cluster if the recovery objective requires surviving a region outage.

Worked example: availability tiers.

```text
one cluster          -> a zone failure interrupts service
two clusters, one region -> zone failure tolerated, region failure not
three clusters, multi-region -> region failure tolerated, higher cost
```

Common mistake: calling two same-region clusters a disaster-recovery plan. It is high availability, not regional DR.

## Can you explain the role of Bigtable's tablet placement policy?

Bigtable's tablet placement policy determines how tablets are assigned to tablet servers to ensure load balancing and efficient resource utilization.

**Mental model.** Placement is service-managed, but app profiles and replication settings constrain where reads are served. You influence placement indirectly through clusters and routing, not through a per-tablet directive.

**How it works.** Bigtable places tablets on nodes to balance load within the available clusters. With replication, each cluster holds a copy, and routing decides which copy answers a request. Cluster groups restrict a profile to a subset of clusters.

**Gotcha.** There is no "pin these rows to that cluster" API. If a dataset must stay in a geography for compliance, use a separate instance in that region rather than a routing hint.

Worked example: what you can and cannot direct.

```text
can:     choose clusters, routing policy, cluster group membership
cannot:  pin a row or tablet to one node or one cluster
```

Common mistake: designing a compliance boundary on an app-profile routing rule alone.

## How does Bigtable handle storage growth over time?

Bigtable automatically scales storage capacity as data grows by adding more servers and tablets to accommodate the increased load.

**Mental model.** Storage grows with rows times versions times cell size. Two levers bound it: fewer versions via GC, and smaller keys and values.

**How it works.** A rough plan is `rows * (key + value bytes) * versions * replication factor`, before compression. GC drops old versions at compaction, so a `max_num_versions` cap is the main control. Replication multiplies stored bytes by the number of clusters.

**Gotcha.** Growth is silent when versions accumulate. A table that writes one row per second with ten retained versions stores ten times what a single-version table stores, and each extra version adds compaction work.

Worked example: a storage estimate.

```text
rows            = 500,000,000
key + value     = 120 bytes
versions kept   = 3
clusters        = 2
raw bytes       = 500,000,000 * 120 * 3 * 2 = 360,000,000,000  (about 335 GiB)
compression     -> actual stored bytes are lower, often by 2x to 5x
```

Common mistake: estimating capacity from live rows alone and forgetting versions and replication.

## Does Bigtable provide data snapshot capabilities?

```bash
# create a backup on a cluster and restore it into a new table
gcloud bigtable backups create orders-2026-10-04 \
  --instance=acme-bt --cluster=acme-bt-c1 --table=orders \
  --retention-period=30d

gcloud bigtable instances tables restore \
  --instance=acme-bt --source=projects/acme-prod/instances/acme-bt/clusters/acme-bt-c1/backups/orders-2026-10-04 \
  --destination=orders_restored --async
```

**Mental model.** A backup is the snapshot concept. There is no copy-on-write table snapshot to branch from; a backup is an independent copy that restores into a new table.

**How it works.** You create a backup from a table, optionally set an expiration, and restore it into a new table in the same or another cluster. Backups are useful before risky migrations and for point-in-time recovery.

**Gotcha.** A backup has no incremental chain to follow. It captures the table at creation time, so a backup taken before a long migration is stale by the end of it.

Worked example: backup before a key migration.

```bash
gcloud bigtable backups create pre-migration \
  --instance=acme-bt --cluster=acme-bt-us-east1 --table=device_readings
# ... migration ...
gcloud bigtable tables restore device_readings_old \
  --instance=acme-bt --cluster=acme-bt-us-east1 \
  --source=projects/acme-prod/instances/acme-bt/clusters/acme-bt-us-east1/backups/pre-migration
```

Common mistake: treating a replication copy as a snapshot to restore from. It has already received the bad write.

## Can you explain how Bigtable handles storage and retrieval of large objects?

It does not. Bigtable has row and cell size limits and is optimized for many small rows, so large objects belong in Cloud Storage. Store the object there and keep a reference (bucket, path, size, checksum) in Bigtable.

**Mental model.** Bigtable can hold large cell values up to a service maximum on the order of 100 MiB, but it is not object storage. Large cells cost cache, slow reads, and travel uncompressed on the wire.

**How it works.** Store a small pointer in Bigtable and the bytes in Cloud Storage when objects exceed a few MiB. The row key gives you the fast metadata lookup; the object store serves the payload. Keep the pointer and the checksum in the cell.

**Gotcha.** A cell that is tens of megabytes evicts useful cache data and makes every read of that row expensive, even if you only need a different column.

Worked example: pointer pattern.

```text
row key:  media#img-7781
family m: gcs_uri = "gs://acme-media/img-7781.jpg"
          size    = "4123456"
          sha256  = "..."
payload:  stored in Cloud Storage, served by a signed URL
```

Common mistake: storing a whole image in a cell and then reading the row for a thumbnail. Fetch only the columns you need, and put the bytes elsewhere.

## How does Bigtable handle schema changes without downtime?

Bigtable supports schema changes without downtime by allowing you to add or remove columns without interrupting the ongoing read and write operations.

**Main point.** Adding or dropping a column family is an online metadata operation. Adding a qualifier is just a write. Neither rewrites existing rows or blocks traffic.

**How it works.** `cbt createfamily` and the client library add a family with a GC policy. Writers begin using new qualifiers immediately. Readers must handle absence because old rows have no cell for the new qualifier.

**Gotcha.** Dropping a family deletes data, and there is no undo without a backup. Confirm with the owners before you remove one.

Worked example: online family add.

```bash
cbt -project acme-prod -instance acme-bt createfamily device_readings telemetry
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings telemetry maxage=7d
```

Common mistake: scheduling a maintenance window for a schema change. There is nothing to take offline.

## Does Bigtable support full-text search capabilities through integrations?

Yes, Bigtable can be integrated with other full-text search engines like Elasticsearch or Apache Lucene for full-text search capabilities.

**Main point.** Not by itself, but integrations make it work. Bigtable is the system of record and a search engine or vector index holds the inverted index.

**How it works.** Change Streams or a Dataflow job mirrors rows into a search engine. The index stores document ids; the application fetches full rows from Bigtable by key. The same pattern backs semantic search when the sink is a vector database.

**Gotcha.** The mirror is eventually consistent and can deliver a change more than once. Index writes must be idempotent, and application reads must tolerate a document that has not landed yet.

Worked example: search over messages.

```text
Bigtable row = conv-7781#9999999999, family m: body = "..."
Change Streams -> Dataflow -> search engine index
query -> search engine returns row keys -> Bigtable point reads
```

Common mistake: indexing in the request path and adding search latency to every write. Do it asynchronously.

## Can you explain how Bigtable handles range scans and filters?

Filters are applied server-side, so a query only returns the cells it needs.

**Mental model.** A scan walks rows in key order over a half-open range `[start, end)`. Filters then drop rows, columns, or cells before they reach the client, but the server still reads the key range you asked for.

**How it works.** Push the prefix into the key range so the scan starts and stops at the right place. Use cell filters to cut columns and versions, and a row limit to bound the result. In Python, `read_rows(start_key=..., end_key=...)` plus `row_filters` does this.

**Gotcha.** A filter is not a seek. Filtering by a value or a non-key column scans the whole range and throws most of it away. Design the key so the range itself is narrow.

Worked example: narrow range plus filters.

```bash
cbt -project acme-prod -instance acme-bt read device_readings prefix=acme#dev-00042# count=10
```

```python
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"acme#dev-00042#",
    end_key=b"acme#dev-00042$",
    limit=10,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
```

Common mistake: scanning a whole table and filtering by `device_id` in the client. Put `device_id` in the key prefix so the server does the narrowing.

In [ ]:
from google.cloud.bigtable import row_filters

# review-only: read the newest version of a few columns in a narrow key range
rows = table.read_rows(
    start_key=b"acme#dev-00042#",
    end_key=b"acme#dev-00042$",
    limit=50,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
rows.consume_all()
for row in rows.rows:
    print(row.row_key.decode(), {c.qualifier.decode(): c.value.decode() for c in row.cells[0]})


## How does Bigtable handle row-level and column-level access control?

IAM handles instance- and table-level permissions; Bigtable has no relational-style row or column grants. Authorized views and separate tables per tenant are the practical ways to narrow what a client can read.

**Mental model.** IAM is table-scoped, not row- or column-scoped. There is no Bigtable equivalent of a row-level security policy or a column grant. Granularity comes from layout and from authorized views.

**How it works.** Separate tables or instances isolate tenants. An authorized view exposes a defined subset of a table to a principal without giving access to the base table. Column-level hiding is done by reading only the families and qualifiers allowed and keeping the rest out of that service.

**Gotcha.** Because a client that can read a table can read every family in it, sensitive columns need their own table or a separate view, not a convention in the read code.

Worked example: splitting sensitive columns.

```text
table device_readings:  telemetry only
table device_pii:       owner_email, address  (separate IAM binding)
```

Common mistake: storing PII and telemetry in one family and trying to restrict it with application code.

## Can you explain the role of Bigtable's compaction strategy in performance optimization?

Bigtable's compaction strategy determines when and how to merge smaller sorted files into larger ones to optimize storage efficiency and read performance.

**Mental model.** Compaction controls how many SSTables a read must check and how much space is reclaimed. Fewer, larger SSTables make reads cheaper; aggressive merging costs background CPU.

**How it works.** The service schedules compacting merges as files accumulate. GC is applied during compaction, so a tighter policy also reduces future read work by removing dead versions. You do not choose a strategy, but you influence the inputs through version count and write rate.

**Gotcha.** Write-heavy tables create flushes faster than compaction can merge if the cluster is undersized, so read latency degrades even though the writes succeed. Node headroom is the protection.

Worked example: versions drive compaction work.

```text
20 writes/s per row, 10 versions kept  -> compaction merges many copies
20 writes/s per row, 1 version kept    -> far less work per merge
```

Common mistake: keeping many versions "just in case" and paying for it in compaction CPU and read amplification.

## Does Bigtable provide automatic indexing for efficient querying?

Not in the relational sense: every row is indexed by its row key automatically, which is the index the storage engine uses for point lookups and range scans, but there is no automatic secondary indexing. For other access patterns, design one yourself with asynchronous secondary indexes, continuous materialized views, or a duplicated row-key layout.

**Main point.** There is one index: the sorted row key. Bigtable does not build secondary indexes on cell values, and it cannot answer an arbitrary predicate without scanning.

**How it works.** Fast paths are point reads and prefix ranges on the key. Everything else is an index table you maintain, a continuous materialized view, an asynchronous secondary index, or a BigQuery copy. Each of these has its own write cost.

**Gotcha.** Adding an index table does not remove the hotspot risk; it copies it. An index on a low-cardinality value creates a small number of very long key prefixes.

Worked example: query paths and their cost.

```text
WHERE rowkey = x            -> point read, fast
WHERE rowkey LIKE 'p%'      -> range scan, fast if p is selective
WHERE status = 'ACTIVE'      -> full scan, or an index table you own
```

Common mistake: promising SQL-like query freedom because a SQL surface exists. The engine still pushes predicates to the row key.

## Can you explain how Bigtable handles large-scale data migration?

Bigtable provides tools and utilities to facilitate large-scale data migration, allowing you to import/export data efficiently.

**Mental model.** Migration is a bulk write problem. Read from the source, write to Bigtable with batched mutations, and shape the key so writes spread across tablets from the start.

**How it works.** Dataflow templates and the Spark connector do bulk import and export. For HBase, offline export followed by import, or online replication from HBase to Bigtable, are the documented paths. Write with your own timestamps if the source has them, and confirm the GC policy keeps them.

**Gotcha.** A migration that writes keys in source order can hotspot. Shuffle or salt the incoming stream, and watch node CPU so compaction keeps up with the import.

Worked example: migration stages.

```text
1. create table + families + GC policy (before importing)
2. export source to Parquet or HFiles
3. bulk import through Dataflow or the Spark connector, keys shuffled
4. verify row counts and checksums, then cut over reads
```

Common mistake: importing first and setting the GC policy after. An `max_age` rule added later can drop the imported history in one compaction.

## How does Bigtable handle concurrent updates to the same cell from multiple clients?

Writes to the same cell are resolved by timestamp: the newest write wins, and stale writes with older timestamps can be rejected by a conditional mutation. This is why optimistic concurrency with a version column is the standard pattern.

**Mental model.** The last write to a cell wins by server timestamp. There is no conflict error and no merge. The losing value becomes an older version until GC drops it.

**How it works.** Each mutation carries a timestamp, either supplied or set by the server. Reads return the highest timestamp. A conditional mutation is the only way to make a concurrent update fail instead of silently overwriting.

**Gotcha.** Clock skew matters when clients supply timestamps. A client with a slow clock can write a value that sorts as older than the data already there, so the new value never surfaces.

Worked example: last-write-wins.

```text
t3.0 client A writes value "42"  (ts 1000)
t3.1 client B writes value "7"   (ts 1001)
read -> "7"; "42" is retained as an older version until GC
```

Common mistake: designing an idempotent import that writes historical timestamps while other writers use server time, then wondering why the import disappears from reads.

## Does Bigtable provide integration with popular business intelligence tools?

BI tools do not query Bigtable directly. The usual pattern is to query Bigtable from BigQuery or to export/stream data into BigQuery or a warehouse, then point Looker or Tableau at that. Treat Bigtable as the serving store, not the BI source.

**Main point.** BI tools talk SQL, and Bigtable is not a SQL warehouse. The standard route is to land Bigtable data in BigQuery (or another warehouse) and point BI at that.

**How it works.** Change Streams plus Dataflow, or a batch Spark export, copies the serving data into a columnar store. BigQuery federation can query Bigtable directly for narrow row-key lookups, but it is not a good fit for dashboard aggregations. Looker, Power BI, and similar tools then read BigQuery.

**Gotcha.** A dashboard that reads Bigtable through federation and aggregates will scan far more rows than a serving workload should, so isolate it or move it to a copy.

Worked example: the serving-to-BI path.

```text
Bigtable -> Change Streams -> Dataflow -> BigQuery -> BI dashboard
```

Common mistake: exposing Bigtable directly to a BI tool and expecting fast `GROUP BY` queries.

## Can you explain the role of Bigtable's Bloom filter in read operations?

Bigtable's Bloom filter is used during read operations to quickly determine whether a requested row or column may exist in a tablet, reducing unnecessary disk I/O.

**Mental model.** A Bloom filter is a probabilistic set test used to skip files that cannot contain a key. It never gives a false negative, so skipping is safe.

**How it works.** For a point read, the service checks file metadata before reading an SSTable. A negative result avoids a disk read and lowers latency. The service manages filter scope and sizing; the HBase `ROW` and `ROWCOL` settings have no Bigtable equivalent.

**Gotcha.** A Bloom filter does nothing for a range scan, because every row in the range must be examined anyway. It helps only when you can name the key.

Worked example: point read versus scan.

```text
point read:  filter skips 9 of 10 SSTables -> one file read
range scan:  every SSTable in the range may hold rows -> filter cannot help
```

Common mistake: attributing fast range scans to Bloom filters. They are fast because the key range is narrow and the data is sorted.

## How does Bigtable handle data consistency across replicas?

Reads are strongly consistent within a cluster. With clusters in multiple regions, updates propagate asynchronously and cross-cluster reads are eventually consistent. Pick the app profile routing that matches the requirement instead of assuming one model everywhere.

**Mental model.** Replication is asynchronous and eventually consistent across clusters. Within one cluster, reads are strongly consistent. A consistency token lets you confirm that a write has reached every replica.

**How it works.** A write commits in the cluster that accepted it. Bigtable then replicates it in the background. `GenerateConsistencyToken` and `CheckConsistency` (or `cbt` equivalents) tell you when all clusters have caught up, which is useful before a migration cutover or a failover.

**Gotcha.** Eventual consistency is not a bug to fix. If the application needs read-your-writes everywhere, use single-cluster routing for that path instead of waiting on replication.

Worked example: checking replica catch-up.

```bash
gcloud bigtable instances tables check-consistency device_readings \
  --instance=acme-bt --consistency-token=AbCdEf123456
```

Common mistake: polling consistency before every read. Check it at explicit boundaries such as failover or migration, not per request.

## Can you explain how Bigtable handles data compression and decompression?

Bigtable uses the Snappy compression algorithm to compress data before storing it on disk. Data is decompressed on-the-fly during read operations.

**Mental model.** Compression is transparent. The service compresses SSTables at rest, decompresses on read, and applies it to everything without a per-table choice.

**How it works.** Because SSTables are sorted by key, neighboring rows often share prefixes and shapes, which compresses well. On a read the node decompresses only the blocks it needs. The client receives plain bytes.

**Gotcha.** Compression lowers storage and read bytes, not client-side payload size. A 2 MiB cell still crosses the network as 2 MiB; compress large values in the application.

Worked example: where bytes are saved.

```text
stored SSTable blocks: compressed (smaller storage, fewer bytes read)
network payload:       uncompressed by the service
```

Common mistake: sizing a cluster on compressed storage and forgetting that wire and CPU cost track logical data.

## Does Bigtable support automatic query optimization?

There is no relational optimizer that rescues a bad access pattern: performance comes from row-key design. Bigtable can also be queried with GoogleSQL for Bigtable, which plans those queries, but for analytical SQL at scale the right move is to query Bigtable from BigQuery rather than treat it as a warehouse.

**Mental model.** There is little to optimize because there is one access path. Bigtable routes a request to the owning tablet node, may use cache and Bloom filters, and streams rows back. The query plan is the key range you choose.

**How it works.** The GoogleSQL surface pushes predicates down to row-key ranges and limits. Client libraries handle retries and routing. Everything else, such as choosing the right prefix or reading fewer families, is your design decision.

**Gotcha.** No optimizer can rescue a query that scans by a non-key field. If a predicate cannot become a key range, the scan reads the whole table.

Worked example: predicate pushdown.

```sql
-- pushes down: becomes a row-key range
SELECT * FROM device_readings WHERE rowkey LIKE 'acme#dev-00042%';
-- does not push down: scans and discards
SELECT * FROM device_readings WHERE temperature = '21.4';
```

Common mistake: assuming the SQL layer has indexes on every column. It does not.

## Can you explain the role of Bigtable's bloom block filter in read operations?

Bigtable's bloom block filter is a probabilistic data structure that helps skip unnecessary disk reads during the lookup process, improving read performance.

**Mental model.** "Bloom block filter" is HBase vocabulary. HBase stores a Bloom filter per HFile or per block to avoid reading blocks that cannot hold a row. Cloud Bigtable keeps the same idea internally but exposes no block-filter setting.

**How it works.** HBase lets you choose `ROW` or `ROWCOL` and a Bloom block size. Cloud Bigtable chooses its own filtering and block layout, so the useful interview point is the concept, not a configuration. Point reads benefit; scans do not.

**Gotcha.** Do not claim a Bigtable table has a user-settable Bloom block filter. The concept exists in the storage engine; the knob does not exist in the API.

Worked example: concept transfer.

```text
HBase:    COMPRESSION + BLOOMFILTER + BLOCKSIZE set per family
Bigtable: none of these are user settings; the service manages them
```

Common mistake: copying an HBase tuning checklist into a Bigtable design review. Most of it has no counterpart.

## How does Bigtable handle backup and restore operations?

Bigtable provides built-in backup and restore functionality, allowing you to create backups, schedule regular backups, and restore data to a specific point in time.

**Mental model.** Backup captures a table at a point in time; restore materializes it as a new table. Backups are independent of the source table and can be copied to another region.

**How it works.** Create a backup on a cluster, optionally with an expiration, and restore it into a new table on the same or another cluster. Restore is not instantaneous for large tables; it is an asynchronous operation you can monitor. Use backups before key redesigns, bulk deletes, and migrations.

**Gotcha.** Backup captures the whole table, not a filtered subset or a key range. If you only need to protect one tenant, that tenant still needs its own table or a separate backup strategy.

Worked example: verify after restore.

```bash
cbt -project acme-prod -instance acme-bt count device_readings_restored
# compare with the source count before you delete the old table
```

Common mistake: deleting the source table as soon as the restore starts. Wait for the operation to finish and for the counts to match.

## Does Bigtable support integration with popular ETL (Extract, Transform, Load) tools?

Yes, Bigtable can integrate with popular ETL tools like Apache Beam, Google Cloud Dataflow, or Apache NiFi for data extraction, transformation, and loading processes.

**Main point.** ETL reaches Bigtable through connectors rather than a native plugin for every tool. Dataflow, Spark, Beam, and Kafka Connect cover the common cases.

**How it works.** Dataflow templates import from Cloud Storage or BigQuery and export to BigQuery or Cloud Storage. The Spark connector reads and writes tables from Spark jobs. The Kafka Connect sink writes Kafka topics into Bigtable. These are batch or streaming jobs, not SQL transforms inside Bigtable.

**Gotcha.** Bulk loads and exports can saturate a cluster. Run large jobs against a replica or with Data Boost, and schedule them away from peak serving hours.

Worked example: pick by direction and latency.

```text
stream Kafka -> Bigtable:      Kafka Connect sink
batch GCS/BigQuery -> Bigtable: Dataflow template
Bigtable -> BigQuery:          Change Streams + Dataflow
Bigtable <-> Spark:             Spark connector
```

Common mistake: running a nightly full export against the production serving cluster at peak QPS.

## Can you explain how Bigtable handles data access control for multi-tenant environments?

Bigtable leverages Google Cloud IAM's multi-tenancy support to enforce fine-grained access control and isolation between tenants.

**Mental model.** Isolation comes from the resource layout, not from row filters. Options, from strongest to weakest, are an instance or table per tenant, an authorized view per tenant, and application filtering.

**How it works.** IAM binds a principal to an instance or table. Authorized views expose a defined subset of a table, which suits a shared table with tenant-scoped keys. For strict isolation, separate tables avoid any shared surface.

**Gotcha.** A shared table with per-tenant row keys still lets a compromised tenant credential read other tenants' rows if it has table access. Match the isolation to the threat model and the compliance requirement.

Worked example: layout by requirement.

```text
regulated tenant data -> instance per tenant
shared SaaS table     -> authorized view per tenant + separate service accounts
internal analytics    -> one table, coarse IAM
```

Common mistake: assuming a row-key prefix acts as a security boundary. It is a performance boundary, not an authorization one.

## How does Bigtable handle data consistency in a multi-region setup?

Replication across clusters is asynchronous, so multi-cluster instances are eventually consistent across regions, while consistency inside a cluster stays strong. App profiles control whether a request goes to a single cluster (consistent reads) or to the nearest replica (lower latency, replicated reads).

**Main point.** Writes replicate asynchronously, so replicas converge within a short window rather than instantly. Single-cluster routing gives strong consistency; multi-cluster routing trades it for locality.

**How it works.** Each cluster accepts reads for its copy. A write reaches one cluster first; the others receive it shortly after. Consistency tokens report when all clusters have caught up. Failover reroutes clients without data loss because every committed write is durable in at least the accepting cluster.

**Gotcha.** Two clients writing the same cell in different regions can both succeed, and the final value depends on timestamp resolution. If that is unacceptable, serialize the writes through a single cluster or a single owner.

Worked example: choosing a routing policy per workload.

```text
user profile reads (latency matters) -> multi-cluster, eventual
account ledger (correctness matters) -> single-cluster, strong
```

Common mistake: one global app profile for a system whose workloads have different consistency needs.

## Can you explain how Bigtable handles schema evolution for existing data?

Bigtable allows you to add or remove columns to the schema without affecting existing data. The new schema will be applied to new writes and subsequent read operations.

**Mental model.** Existing rows are not rewritten when the schema changes. New qualifiers simply appear on new or updated rows, and old rows keep returning nothing for them.

**How it works.** Because a cell is addressed by qualifier, a reader that requests a missing qualifier gets no cell, which the code maps to a default. If a type changes, both encodings can coexist, so version the qualifier name instead of reusing it.

**Gotcha.** There is no server-side migration or backfill. If analytics needs the new field on historical rows, a separate batch job must add it, and that job competes with serving writes.

Worked example: versioning a changed encoding.

```text
old: profile: status      = "1"
new: profile: status_v2   = "ACTIVE"
read: prefer status_v2, fall back to status
```

Common mistake: changing the meaning of a qualifier in place. Historical rows silently disagree with the new code.

## Does Bigtable provide support for complex data types like arrays or JSON?

Bigtable stores data as byte arrays, which allows you to store complex data types like arrays or JSON by serializing them into byte representations.

**Main point.** Values are bytes. You can store JSON text, a serialized protobuf, or a packed binary encoding, but the server does not parse or index it.

**How it works.** Store JSON in a cell when the payload is read whole and never filtered in Bigtable. Use separate qualifiers for fields you filter or aggregate outside the store. Protobuf schemas with schema bundles give typed access and validation at the application boundary.

**Gotcha.** A JSON blob is opaque to GC, filters, and the SQL surface. To query inside it, either project the fields into qualifiers at write time or land the JSON in BigQuery and use its JSON functions.

Worked example: opaque versus projected.

```text
opaque:    r:payload = {"temp":21.4,"hum":48,"ts":"..."}
projected: r:temperature = "21.4", r:humidity = "48"
```

Common mistake: storing JSON and then trying to filter on a field inside it with a row filter. Row filters match bytes, not JSON paths.

## Can you explain how Bigtable handles access control for different levels of data granularity?

IAM grants apply at instance and table level, and to backups as resources. Row-key-prefix and column granularity comes from authorized views rather than from IAM alone.

**Main point.** The grantable resources are project, instance, and table. There is no column or cell grant. Granularity below the table is a layout choice.

**How it works.** Roles such as `roles/bigtable.reader`, `roles/bigtable.user`, and `roles/bigtable.admin` bind at project, instance, or table scope. Authorized views add a managed way to share a subset of a table. Sensitive columns belong in a table with its own bindings.

**Gotcha.** A reader on a table reads every family and every row in it. Do not describe family-level or column-level IAM as a Bigtable feature; it is not one.

Worked example: scoping a grant.

```bash
gcloud bigtable instances tables add-iam-policy-binding device_readings \
  --instance=acme-bt \
  --member="serviceAccount:svc@acme.iam.gserviceaccount.com" \
  --role="roles/bigtable.reader"
```

Common mistake: granting at the project level when a table-level binding is enough. Broad grants are hard to audit and to revoke.

## How does Bigtable handle data distribution across different availability zones within a region?

Bigtable automatically distributes data across different availability zones within a region to ensure high availability and fault tolerance.

**Main point.** A cluster lives in one zone. Two clusters in two zones of the same region give you in-region high availability and automatic failover.

**How it works.** Replication copies tables between the clusters, and a multi-cluster app profile routes around a zone failure. The storage layer in Colossus is itself replicated, so durable data survives a zone loss even before failover completes. The remaining cluster serves traffic at reduced capacity until you add nodes.

**Gotcha.** Two zones double the cluster cost and add cross-zone replication traffic even when nothing fails. Size each zone to carry the full load if you want seamless failover, not just survival.

Worked example: sizing for zone failover.

```text
steady state: zone A 6 nodes, zone B 6 nodes
zone A fails: zone B 6 nodes must serve everything
plan:         size each cluster for peak, not for half of peak
```

Common mistake: splitting a node budget in half across two zones and assuming failover will hold peak traffic.

## Can you explain the role of Bigtable's read-modify-write operation?

Bigtable's read-modify-write operation allows you to read data, modify it, and write it back atomically within a single transaction, ensuring consistency.

**Mental model.** Read-modify-write is an application pattern, not a server transaction. You read a row, compute a new value, and write it back. Bigtable gives you the tool to make it safe: a conditional mutation.

**How it works.** Put the expected current state in a predicate filter and the new state in the mutation. `CheckAndMutateRow` applies the write only if the predicate matches. If it does not, re-read and retry with backoff. For counters, use an aggregate family or a sharded counter to avoid contention.

**Gotcha.** An unconditional read-modify-write loses updates under concurrency. The wider the gap between the read and the write, the more likely the loss.

Worked example: safe update with a version guard.

```python
row = table.conditional_row(b"acct-7781")
row.set_cell("s", b"balance", str(new_balance).encode())
row.set_cell("s", b"version", str(version + 1).encode())
row.commit()
```

Common mistake: retrying the whole operation without re-reading. The predicate must compare against the value you actually read.

## Does Bigtable support integration with machine learning frameworks like TensorFlow or PyTorch?

Bigtable is a common online feature store because lookups are fast and predictable by row key. It is rarely the training data source: training reads large volumes sequentially, which belongs in BigQuery or Parquet on object storage, while the serving path reads features from Bigtable.

**Main point.** Bigtable is a feature store for serving, not a training data store. Training reads Parquet or BigQuery in bulk; serving reads one feature row by key.

**How it works.** Export or stream features to a columnar store for training, and keep the online copy in Bigtable. A model server reads a feature row by entity key at prediction time, in single-digit milliseconds. Vertex AI and similar platforms can consume Bigtable as a feature source.

**Gotcha.** Training on the online store creates training/serving skew if the transformation differs. Define the feature once and apply the same logic on both paths, or materialize features in one job that feeds both.

Worked example: online and offline from one definition.

```text
offline: BigQuery/Parquet -> train model
online:  Bigtable point read by entity key -> score in the request
```

Common mistake: pointing a training job at Bigtable and scanning the serving table, which competes with production reads and drifts from the batch definition.

## Can you explain the role of Bigtable's mutation operations in write operations?

Bigtable's mutation operations allow you to specify modifications to be applied during write operations, such as inserting or updating data in specific cells.

**Mental model.** A mutation is a set of changes to one row: set a cell, delete a cell, delete a version, or delete a family's cells. Mutations are batched and applied atomically per row.

**How it works.** The client builds a row object, sets cells with optional timestamps, and commits. `mutate_rows` sends many rows in one batch for throughput but is not atomic across rows. Conditional rows add a predicate to the mutation.

**Gotcha.** Setting a cell always appends a version; it does not replace history. Without a version cap, repeated writes to the same cell grow storage and compaction work.

Worked example: one atomic row mutation.

```python
row = table.direct_row(b"acme#counter")
row.set_cell("c", b"value", b"1")
row.delete_cell("c", b"temp")
row.commit()
```

Common mistake: assuming `mutate_rows` is transactional. It is a throughput tool; partial failure is possible and retries must be idempotent.

## How does Bigtable handle access control for different types of operations, such as read, write, or delete?

IAM defines read, write, and admin permissions at instance and table level. Authorized views then narrow an application's data access to a row-key prefix and a column set when table-level access is too broad.

**Main point.** IAM roles bundle operations. There is no separate permission for read versus delete at the table level; if a principal can write, it can also delete cells through the same data API.

**How it works.** `roles/bigtable.reader` allows reads. `roles/bigtable.user` allows reads and writes to data. `roles/bigtable.admin` manages tables, clusters, and IAM. Deletes are data mutations, so a writer can delete. Audit logging records what happened.

**Gotcha.** If delete must be restricted but write allowed, the split has to be in the application or in the data layout, because IAM cannot express it. Consider append-only tables with GC, where no client has delete rights.

Worked example: role to capability.

```text
bigtable.reader -> read rows
bigtable.user   -> read and write rows (including deletes)
bigtable.admin  -> manage tables, clusters, backups, IAM
```

Common mistake: telling an interviewer that Bigtable has a distinct delete permission separate from write.

## Can you explain how Bigtable handles data replication across regions in terms of consistency and latency?

Bigtable replicates data asynchronously across regions, which may result in eventual consistency and varying levels of latency between regions.

**Main point.** Cross-region replication lowers read latency by putting a copy near users and raises write latency slightly because writes still commit in one cluster before propagating. Consistency is eventual across regions.

**How it works.** A multi-cluster app profile routes a read to the nearest healthy cluster, which returns its local copy immediately. Replication lag is usually small but non-zero. A consistency token reports when all clusters have the write.

**Gotcha.** Sizing each cluster for full load is part of the design. If failover sends all traffic to one region and that cluster was sized for half, latency rises sharply right when you need it.

Worked example: latency shape.

```text
read from nearest replica: single-digit ms, may be slightly stale
write from a distant client: crosses to the accepting cluster, then replicates
```

Common mistake: promising globally consistent reads with multi-cluster routing. That requires single-cluster routing and gives up the latency win.

## How does Bigtable handle data durability and fault tolerance?

Bigtable ensures data durability and fault tolerance through replication, storing multiple replicas of data across different servers and regions.

**Mental model.** Durability comes from replicating the commit log and SSTables in Colossus. Fault tolerance comes from stateless nodes and automatic tablet reassignment. A node loss is an availability event, not a data-loss event.

**How it works.** A write is durable once it is in the commit log, which is replicated by the storage layer. If a node dies, the service reassigns its tablets and replays from the log. Replication to another cluster protects against a full-cluster or region outage.

**Gotcha.** Durability does not protect against logical errors. A `drop` of a table or a bad migration is replicated everywhere; only a backup recovers it.

Worked example: failure to outcome.

```text
node crash      -> tablets reassigned, no data loss
zone outage     -> second cluster serves, no data loss
bad deploy      -> data loss unless a backup exists
```

Common mistake: equating durability with recoverability. Keep tested backups for the logical failure modes.

## Can you explain how Bigtable handles high availability and seamless failover?

Bigtable provides high availability through replication and automatic failover mechanisms, ensuring continuous access to data even in case of server or region failures.

**Main point.** Availability comes from replication across zones or regions plus an app profile that routes around failure. Clients retry against the healthy cluster, so failover is a routing change rather than an operator action.

**How it works.** A multi-cluster app profile detects an unhealthy cluster and sends new requests elsewhere. Writes committed in the failed cluster are already replicated, so no committed data is lost. Single-cluster profiles do not fail over automatically; they are pinned for consistency.

**Gotcha.** Seamlessness depends on capacity. If the surviving cluster cannot carry peak load, failover succeeds technically but the application slows down. Size for the loss of one cluster.

Worked example: failover readiness.

```text
policy: multi-cluster routing, 2 clusters
sizing: each cluster can serve peak alone
test:   fail a cluster during a game day and watch p99
```

Common mistake: assuming a replicated cluster is a hot standby at full capacity without ever testing a failover.

## Does Bigtable support time travel queries with fine-grained control over historical data retrieval?

Yes, Bigtable supports time travel queries, allowing you to retrieve specific versions of data based on timestamps or time ranges.

**Main point.** Not in the warehouse sense. There is no `AS OF` timestamp for a table. Historical reads are cell versions bounded by the GC policy, plus table-level backup and restore.

**How it works.** A read can return multiple versions or filter by a timestamp range, so recent history within the retention window is available. `max_num_versions` and `max_age` define that window. Backups provide coarser point-in-time recovery at the table level.

**Gotcha.** GC is enforced at compaction, so the effective history can be shorter than the rule suggests, and it is never longer. If history is a product requirement, write an append-only history table or stream changes to a warehouse.

Worked example: what "history" can mean.

```text
within GC window: read versions of a cell
older than window: gone, unless a backup or an exported copy exists
```

Common mistake: building an audit feature on cell versions. It silently loses data when compaction runs.

## Can you explain how Bigtable handles data partitioning and load balancing?

Bigtable partitions data by range partitioning the row keys, and it automatically balances the distribution of tablets across tablet servers to ensure load balancing.

**Mental model.** Partitioning is automatic and data-driven: tablets split on size, and load balancing moves tablets to even out request load. There is no fixed partition count to tune.

**How it works.** The service watches per-node and per-tablet load, splits growing tablets, merges small ones, and reassigns tablets to underloaded nodes. Clients refresh their cached key-range map as boundaries move.

**Gotcha.** Load balancing cannot create parallelism inside one row. A single hot key or a single hot prefix remains a bottleneck no matter how the tablets move.

Worked example: partition follows keys.

```text
keys: 00#.. through 15#..   -> 16 prefix ranges, spread across nodes
keys: 2026-10-04T.. only    -> one growing range, one hot end
```

Common mistake: treating the tablet count as a tunable like a Kafka partition count. It is an outcome, not an input.

## How does Bigtable handle access control for data in transit?

Bigtable encrypts data in transit using industry-standard encryption protocols, ensuring secure communication between clients and the Bigtable service.

**Main point.** Client connections use TLS by default. IAM authenticates the caller, TLS protects the bytes, and audit logs record the access. There is no plaintext data API endpoint.

**How it works.** The client libraries negotiate TLS to the service. Application Default Credentials or a service account supplies identity. Private Google Access or VPC Service Controls can restrict the network path further. Admin and data API calls are both authenticated and authorized.

**Gotcha.** TLS protects the path, not the destination. A service account key copied to a laptop defeats the transport protection, so manage keys with workload identity and rotate them.

Worked example: layers in the request path.

```text
network: TLS (and optionally VPC Service Controls)
identity: service account or ADC
authorization: IAM role on the instance or table
record: Data Access audit logs when enabled
```

Common mistake: treating transport encryption as sufficient and leaving a broad project-level role in place.

## Can you explain the role of Bigtable's client-side buffering and batching in optimizing write operations?

Bigtable's client-side buffering and batching allow you to group multiple write operations together before sending them to the server, reducing network overhead and improving write performance.

**Mental model.** Batching amortizes network and RPC overhead across many mutations. Buffering groups small writes so fewer, larger requests reach the service.

**How it works.** `mutate_rows` sends a batch of rows in one call. The client library batches within limits and retries failed entries. Larger batches raise throughput up to a point, then latency and memory use grow and a retry re-sends more work.

**Gotcha.** Bigger is not always better. A huge batch can time out, and a retry can re-apply mutations unless they are idempotent. Tune batch size against tail latency, not just throughput.

Worked example: batch by count and size.

```python
table.mutate_rows(rows)   # rows is a list; the library chunks the call
# keep each row small and each batch to a few hundred rows, then measure p99
```

Common mistake: buffering writes in the application for seconds to build a big batch. That adds latency and risks losing the buffer on a crash; the service already has a commit log.

## How does Bigtable handle data replication and failover in a multi-region setup with data consistency requirements?

In a multi-region setup, Bigtable replicates data across regions and provides automatic failover mechanisms to ensure data consistency and high availability, maintaining replicas across regions in sync.

**Main point.** Put the consistency-critical workload on single-cluster routing and the latency-sensitive reads on multi-cluster routing, using separate app profiles on the same replicated instance.

**How it works.** The instance replicates to every cluster. A serving profile routes reads to the nearest cluster and tolerates eventual consistency. A ledger profile pins writes and reads to one cluster for read-your-writes. A failover moves the pinned profile only during an incident, deliberately.

**Gotcha.** After a failover of a pinned profile, clients can briefly read stale data from the new cluster until replication catches up. Check consistency before cutting over, and expect a short write pause.

Worked example: two profiles, one instance.

```text
instance acme-bt: clusters us-east1, europe-west1
app profile "serving": multi-cluster routing
app profile "ledger":  single-cluster acme-bt-us-east1
failover:              repoint "ledger" after checking consistency
```

Common mistake: one profile for everything. It forces a choice between slow global reads and inconsistent critical writes, when the system needs both behaviours.

## Tuning row keys for even load and locality

**Mental model.** Two goals pull in opposite directions. Even load wants writes spread across the whole key space. Locality wants the rows one query needs to sit next to each other. A good key satisfies both for the queries that matter, and accepts a trade-off for the rest.

**How it works.** Diagnose first. Key Visualizer shows per-prefix traffic; a bright horizontal band on a prefix means a hotspot. Then choose a fix: field promotion moves a high-cardinality field to the front, salting prepends a fixed-width bucket, hashing spreads a single hot key, and a reversed timestamp turns "latest first" into a prefix scan.

**Gotcha.** Salting destroys free range scans over the salted field, because the salt reorders rows. Add it only to the leading, hottest part of the key, and keep the rest sorted.

Worked example: read the symptom, then fix the key.

```text
symptom: writes all land on the last tablet, node CPU is pegged
cause:   key = 2026-10-04T12:00:00Z#device-42 (ascending timestamp first)
fix:     key = 07#device-42#9999999999 (16 salt buckets, reversed timestamp)

symptom: one user row takes most reads, one tablet is hot
cause:   key = CONFIG#global-flags
fix:     key = 01#CONFIG#global-flags plus an application cache
```

A hotspot check is simple arithmetic: compute the share of writes per key prefix and compare the largest share to an even split.

```python
from collections import Counter

counts = Counter(k.split("#")[0] for k in ["00#d1", "00#d2", "00#d3", "03#d4", "03#d5"])
total = sum(counts.values())
worst = max(counts.values()) / total
print(f"worst prefix share = {worst:.0%} across {len(counts)} prefixes")
```

Common mistake: salting and then querying the old unsalted prefix. Reads must use the same fan-out as writes.

## Tuning GC policies and version counts

**Mental model.** GC policy is a storage-versus-read-cost dial. More retained versions mean more bytes, more compaction work, and slower wide reads. Fewer versions mean cheaper storage and faster reads, but less history. Set the rule per column family from the retention the product actually needs.

**How it works.** `max_num_versions` caps versions per cell. `max_age` caps age. `union` keeps a cell if either rule keeps it, so it is the permissive combination. `intersection` keeps a cell only if both keep it, so it is the restrictive one. The rule is enforced at compaction.

**Gotcha.** The default policy is not a storage cap. A table that appends to the same cell every second can retain far more than expected until you set a version limit.

Worked example: choose per family, not per table.

```text
family profile: {"max_num_versions": 1}          # only current state, cheapest
family events:  {"max_age": "2592000s"}          # 30 days of history
family audits:  {"max_num_versions": 12}         # last 12 changes, no age cap
combined:       {"union": {"rules": [{"max_num_versions": 3}, {"max_age": "2592000s"}]}}
```

```bash
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings events maxage=30d
cbt -project acme-prod -instance acme-bt setgcpolicy device_readings profile maxversions=1
cbt -project acme-prod -instance acme-bt ls device_readings
```

Common mistake: using `union` to try to cap storage. Union keeps cells that any rule keeps, so it retains more; use `intersection` to bound bytes.

## Tuning cluster sizing, nodes, and app profiles

**Mental model.** Nodes buy throughput; storage grows on its own. Size the cluster from peak requests per second, add headroom for compaction, and use app profiles to route that traffic to the right cluster with the right consistency.

**How it works.** Plan on the documented per-node ceiling of roughly 10,000 simple reads or writes per second, and target well below it (about 60 to 70 percent) so compaction and spikes have room. Autoscaling adjusts between a minimum and maximum against a target utilization. Choose SSD for latency-sensitive workloads. An app profile selects single-cluster routing (read-your-writes) or multi-cluster routing (nearest replica, eventual consistency), and can be limited to a cluster group.

**Gotcha.** Read and write load share the same nodes, so take the larger of the two estimates, not the sum. And size every cluster for full peak if you expect seamless failover.

Worked example: node count from peak throughput.

```text
peak reads  = 50,000/s   peak writes = 20,000/s
usable per node = 10,000 * 0.7 = 7,000/s
read nodes  = ceil(50,000 / 7,000) = 8
write nodes = ceil(20,000 / 7,000) = 3
cluster     = 8 nodes (the larger requirement)
```

```bash
cbt -project acme-prod -instance acme-bt createcluster acme-bt-eu \
  --zone=europe-west1-b --num-nodes=8
cbt -project acme-prod -instance acme-bt createappprofile acme-bt eu-serving \
  --route-any
cbt -project acme-prod -instance acme-bt createappprofile acme-bt ledger \
  --route-to=acme-bt-us-east1
```

Common mistake: sizing on average QPS. Peak-to-average ratios above three are normal in serving workloads.

In [ ]:
from google.cloud import bigtable
from google.cloud.bigtable import enums

# review-only: choose routing with an app profile, then tune nodes per cluster
admin = bigtable.Client(project="acme-prod", admin=True)
instance = admin.instance("acme-bt")

instance.app_profile(
    "eu-serving",
    routing_policy=enums.RoutingPolicyType.ANY,   # nearest healthy cluster
    description="latency-first reads",
).create()

instance.app_profile(
    "ledger",
    routing_policy=enums.RoutingPolicyType.SINGLE,
    description="read-your-writes on one cluster",
).create()


## Tuning reads and writes: batching, filters, and caching

**Mental model.** Reads are cheapest when they are point lookups or narrow key ranges that ask for few versions and few columns. Writes are cheapest when the key spreads them and GC bounds the versions they create. Everything else is overhead you can design away.

**How it works.** Bound every scan with a start key, an end key, and a limit. Use filters to cut columns and versions: a cells-per-column limit, a column-qualifier regex, a timestamp range. Batch writes with `mutate_rows`, but keep batches small enough to avoid timeouts and make retries idempotent. For very large read jobs, use Data Boost or a replica so serving traffic is untouched.

**Gotcha.** A row filter is applied after the server reads the key range. Filtering on a value does not narrow the scan; only the key range does.

Worked example: bound reads, then federate for analytics.

```python
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"acme#dev-00042#",
    end_key=b"acme#dev-00042$",
    limit=50,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
rows.consume_all()
```

For analytical questions, query Bigtable from BigQuery through an external table and keep a row-key filter so predicates push down.

```sql
SELECT rowkey, MAX(temperature) AS max_temp
FROM analytics.device_readings
WHERE rowkey LIKE 'acme#dev-00042%'
GROUP BY rowkey;
```

Common mistake: reading a wide row and filtering columns in the application. A qualifier filter drops those bytes before they cross the network.

## Real world use cases for Bigtable

### 1. IoT telemetry serving

Millions of devices write readings every few seconds; the application shows the latest readings per device and a downsampled history.

In [ ]:
from google.cloud import bigtable
from google.cloud.bigtable import column_family

client = bigtable.Client(project="acme-prod", admin=True)
table = client.instance("acme-bt").table("device_readings")
table.create()

# expire readings automatically after 7 days
cf = table.column_family("r", gc_rule=column_family.MaxAgeGCAgeRule(7 * 24 * 60 * 60))
cf.create()

# device prefix plus a reversed timestamp so the newest reading sorts first
row = table.direct_row(b"dev-00042#2026-10-04T12:00:00Z")
row.set_cell("r", b"temperature", b"21.4")
row.set_cell("r", b"humidity", b"48")
row.commit()

```bash
# the same operations from the cbt CLI
cbt -project acme-prod -instance acme-bt createtable device_readings
cbt -project acme-prod -instance acme-bt createfamily device_readings r
cbt -project acme-prod -instance acme-bt set device_readings dev-00042#2026-10-04T12:00:00Z r:temperature=21.4
cbt -project acme-prod -instance acme-bt read device_readings prefix=dev-00042#
```

Why Bigtable: millisecond key lookups at very high write rates, with TTL enforced by garbage collection instead of a cleanup job. Salt or bucket the key prefix when one device or tenant dominates traffic.

### 2. User activity and message history

A messaging product stores per-conversation history and a per-user inbox. Reads are prefix scans ordered by time.

In [ ]:
from google.cloud.bigtable import row_filters

rows = table.read_rows(
    start_key=b"conv-7781#",     # newest first because the timestamp is reversed in the key
    end_key=b"conv-7781$",
    limit=50,
    filter_=row_filters.CellsColumnLimitFilter(1),
)
rows.consume_all()

Why Bigtable: a sorted row key gives "the last 50 messages" with a range scan and no secondary index, while appends stay cheap. Model the "all conversations for a user" query as a second row-key layout or an asynchronous secondary index.

### 3. Online feature store for ML serving

A fraud model needs 30-day aggregates per card at prediction time.

```text
row key: card#c-9182#feature#velocity_24h
columns: f:value, f:updated_at
write:   conditional mutation so an older write cannot overwrite a newer value
read:    one row per feature at scoring time (sub-10 ms p99)
train:   export or stream the same features to BigQuery for offline training
```

Why Bigtable: the serving path needs predictable latency at high QPS, which is exactly a row-key lookup. Do not scan Bigtable to build training sets - use BigQuery or Parquet, and keep both paths fed by one feature definition to avoid training/serving skew.

### 4. Real-time bidding and profile lookup

Ad tech keeps a user profile that every bid request reads, and reads must be fast in every region.

```text
row key:  bucket#user_id          (bucket spreads hot users across tablets)
families: profile (small, rarely changing), events (append-only, GC by age)
routing:  multi-cluster app profile so bids read the nearest replica
```

Why Bigtable: high read throughput with replicated low-latency reads. Multi-cluster routing accepts eventual consistency for profile updates, which bidding tolerates, in exchange for serving reads close to the user.

### 5. Operational serving plus BigQuery analytics

The same data serves the application and the analyst. Change Streams feed the analytical copy instead of a dual write.

```text
Bigtable (serving) --Change Streams--> Dataflow --> BigQuery (analytics) --> BI
                          |
                   Data Boost for large read jobs without competing with serving traffic
```

```sql
-- analytical questions run in BigQuery, not against the serving table
SELECT DATE(event_ts) AS ds,
       COUNT(*) AS events,
       APPROX_QUANTILES(latency_ms, 100)[OFFSET(95)] AS p95
FROM analytics.events_enriched
GROUP BY ds
ORDER BY ds;
```

Why Bigtable: the serving table stays optimized for key lookups, analytics runs on a columnar copy, Change Streams remove the fragile dual write, and Data Boost isolates heavy scans from production traffic.

**Mental model.** Work backward from the access pattern. Write down the top reads and writes, their latency targets, and their request rates. If the answers are "by key" or "by a narrow ordered range" at high rate, Bigtable fits.

**How it works.** The five cases below all use a key-first design, per-family GC for different retention, and app profiles for routing or read isolation. Each one pairs Bigtable with an analytical copy so heavy scans never touch the serving path.

**Gotcha.** A use case is not a schema. Two teams can both say "IoT telemetry" and need different keys, different families, and different GC rules, depending on whether they query by device, by site, or by time window.

Common mistake: copying a case study's key design without its query list. The key is a function of the queries, not of the industry.

The pattern to reuse across all five cases:

```text
serving reads and writes  -> Bigtable, key-first, per-family GC
analytics and dashboards   -> BigQuery, fed by Change Streams or export
large read jobs            -> Data Boost or a replica, never the serving path
routing and consistency    -> app profiles, one per workload
```

## Bigtable exercises

These exercises use only the Python standard library and model Bigtable key, sizing, and GC decisions with small runnable calculations. Each one states the task, gives a hint, and names the expected result before the code cell. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are engineering arithmetic, not benchmarks: bytes and versions for storage, ratios for hotspot and selectivity checks. Treat the asserted values as a check on the formula.

### Try it 1. Design a row key from access patterns

**Task.** Build a row key for the queries "latest reading for one device, newest first" and "all readings for a device in a time window". The key must be `salt#device_id#reverse_ts` with a two-digit salt, and the reversed timestamp must sort newest first.

**Hint.** Use a far-future sentinel and subtract the epoch milliseconds: `9_999_999_999_999 - ts_ms`. Format with `%013d`, and derive the salt with `hashlib.md5(device_id.encode())` modulo the bucket count.

**Expected result.** Every key starts with a two-digit salt below 16, keys for one device sort newest first, and a prefix scan returns them in that order.

In [ ]:
import hashlib

BUCKETS = 16
MAX_TS = 9_999_999_999_999


def row_key(device_id, ts_ms):
    salt = int(hashlib.md5(device_id.encode()).hexdigest(), 16) % BUCKETS
    return f"{salt:02d}#{device_id}#{MAX_TS - ts_ms:013d}"


base = 1_700_000_000_000
timestamps = [base + offset for offset in (120_000, 60_000, 0)]  # newest first
keys = [row_key("dev-00042", ts) for ts in timestamps]
for key in keys:
    print(key)

salts = [key.split("#")[0] for key in keys]
print("distinct salts:", sorted(set(salts)))
print("newest first:", keys == sorted(keys))

assert all(len(salt) == 2 and 0 <= int(salt) < BUCKETS for salt in salts)
assert keys == sorted(keys)
assert keys[0].endswith(f"{MAX_TS - (base + 120_000):013d}")
assert len(set(keys)) == 3
print("ok")


### Try it 2. Compute storage growth from rows, versions, and cell size

**Task.** A table has 500 million rows, an average key plus value size of 120 bytes, keeps 3 versions per cell, and replicates to 2 clusters. Compute raw stored bytes, the GiB figure, and the bytes after a 3x compression ratio.

**Hint.** Raw bytes are `rows * cell_bytes * versions * clusters`. Compression divides by the ratio. Use `1024 ** 3` for GiB and round only when printing.

**Expected result.** `360,000,000,000` raw bytes, about `335.3 GiB`, and `120,000,000,000` compressed bytes.

In [ ]:
ROWS = 500_000_000
CELL_BYTES = 120
VERSIONS = 3
CLUSTERS = 2
COMPRESSION_RATIO = 3.0

raw_bytes = int(ROWS * CELL_BYTES * VERSIONS * CLUSTERS)
compressed_bytes = int(raw_bytes / COMPRESSION_RATIO)

print(f"raw bytes        = {raw_bytes:,}")
print(f"raw GiB          = {raw_bytes / 1024 ** 3:.1f}")
print(f"compressed bytes = {compressed_bytes:,}")

assert raw_bytes == 360_000_000_000
assert round(raw_bytes / 1024 ** 3, 1) == 335.3
assert compressed_bytes == 120_000_000_000
print("ok")


### Try it 3. Estimate node count from throughput targets

**Task.** Peak load is 50,000 reads per second and 20,000 writes per second. A node handles about 10,000 simple reads or writes per second, and you plan to use 70 percent of that. Compute the nodes needed for reads, for writes, and the cluster size.

**Hint.** Use `math.ceil` on `peak / (per_node * target_utilisation)`. Reads and writes share nodes, so the cluster needs the larger of the two numbers, not the sum.

**Expected result.** 8 read nodes, 3 write nodes, and a cluster of 8 nodes.

In [ ]:
import math

PEAK_READS = 50_000
PEAK_WRITES = 20_000
PER_NODE = 10_000
TARGET = 0.7

usable_per_node = PER_NODE * TARGET
read_nodes = math.ceil(PEAK_READS / usable_per_node)
write_nodes = math.ceil(PEAK_WRITES / usable_per_node)
cluster_nodes = max(read_nodes, write_nodes)

print(f"usable per node = {usable_per_node:,.0f} operations/s")
print(f"read nodes      = {read_nodes}")
print(f"write nodes     = {write_nodes}")
print(f"cluster nodes   = {cluster_nodes}")

assert usable_per_node == 7_000.0
assert read_nodes == 8
assert write_nodes == 3
assert cluster_nodes == 8
print("ok")


### Try it 4. Size a scan from row count and filter selectivity

**Task.** A range scan covers 8 million rows of 200 bytes each. A qualifier filter keeps 1 column in 40, and a version filter keeps 1 version in 5. Compute rows returned, bytes read before filters, and bytes returned after filters.

**Hint.** Rows returned is `rows * column_selectivity * version_selectivity`. Bytes before filters use the full row count; bytes after use the returned rows. Use integer division only when the arithmetic is exact.

**Expected result.** 40,000 rows returned, `1,600,000,000` bytes read before filters, and `8,000,000` bytes returned.

In [ ]:
ROWS = 8_000_000
ROW_BYTES = 200
COLUMN_SELECTIVITY = 1 / 40
VERSION_SELECTIVITY = 1 / 5

rows_returned = int(ROWS * COLUMN_SELECTIVITY * VERSION_SELECTIVITY)
bytes_before = ROWS * ROW_BYTES
bytes_after = rows_returned * ROW_BYTES
reduction = bytes_before // bytes_after

print(f"rows scanned    = {ROWS:,}")
print(f"rows returned   = {rows_returned:,}")
print(f"bytes before    = {bytes_before:,}")
print(f"bytes returned  = {bytes_after:,}")
print(f"reduction       = {reduction}x")

assert rows_returned == 40_000
assert bytes_before == 1_600_000_000
assert bytes_after == 8_000_000
assert reduction == 200
print("ok")


### Try it 5. Detect a hotspot from key-prefix distribution

**Task.** Given request counts per key prefix, compute the total, the even share, the busiest prefix, its ratio to the even share, and the prefixes above twice the even share.

**Hint.** The even share is `total / len(counts)`. The ratio is `count / even_share`. List comprehension over the counts finds the offenders.

**Expected result.** Total `134,500`, even share `16,812.5`, busiest `00` at about `2.97x`, and `["00"]` above the 2.0x threshold.

In [ ]:
counts = {
    "00": 50_000,
    "01": 20_000,
    "02": 18_000,
    "03": 12_000,
    "04": 10_000,
    "05": 9_000,
    "06": 8_000,
    "07": 7_500,
}

total = sum(counts.values())
even_share = total / len(counts)
ratios = {prefix: count / even_share for prefix, count in counts.items()}
busiest = max(ratios, key=ratios.get)
hot = [prefix for prefix, ratio in ratios.items() if ratio > 2.0]

print(f"total requests = {total:,}")
print(f"even share     = {even_share:,.1f}")
print(f"busiest        = {busiest} at {ratios[busiest]:.2f}x")
print("prefixes above 2.0x:", hot)

assert total == 134_500
assert even_share == 16_812.5
assert round(ratios[busiest], 2) == 2.97
assert hot == ["00"]
print("ok")


### Try it 6. Plan a GC policy from retention days and versions

**Task.** A family must keep 30 days of history but never more than 5 versions per cell. Write a function that classifies a proposed policy as `union`, `intersection`, or `single`, and return the tighter combination for the requirement.

**Hint.** `union` keeps a cell if any rule keeps it. `intersection` keeps it only if all rules keep it, which is what "no more than 5 versions AND no older than 30 days" means. Convert days to seconds with `days * 86_400`.

**Expected result.** The combined policy is `intersection`, `max_age_seconds` is `2,592,000`, and the version cap is `5`.

In [ ]:
def plan_gc(retention_days, max_versions):
    if retention_days and max_versions:
        policy = "intersection"
    elif retention_days or max_versions:
        policy = "single"
    else:
        policy = "none"
    return {
        "policy": policy,
        "max_age_seconds": retention_days * 86_400 if retention_days else None,
        "max_num_versions": max_versions,
    }


plan = plan_gc(retention_days=30, max_versions=5)
print(plan)

assert plan["policy"] == "intersection"
assert plan["max_age_seconds"] == 2_592_000
assert plan["max_num_versions"] == 5
assert plan_gc(0, 3)["policy"] == "single"
assert plan_gc(7, 0)["max_age_seconds"] == 604_800
assert plan_gc(0, 0)["policy"] == "none"
print("ok")


### Try it 7. Model salting fan-out

**Task.** A single hot key takes 64,000 writes per second. Model fan-out with 4, 8, and 16 salt buckets, and pick the smallest bucket count that keeps each salted key under 10,000 writes per second.

**Hint.** Writes per bucket are `total / buckets`. Iterate the candidate bucket counts in ascending order and stop at the first that satisfies the limit.

**Expected result.** 16,000 then 8,000 then 4,000 writes per bucket, and a chosen fan-out of 8 buckets.

In [ ]:
TOTAL_WRITES = 64_000
PER_KEY_LIMIT = 10_000

fan_out = {}
for buckets in (4, 8, 16):
    per_key = TOTAL_WRITES / buckets
    fan_out[buckets] = per_key
    print(f"{buckets:>2} buckets -> {per_key:,.0f} writes/s per salted key")

chosen = next(b for b in (4, 8, 16) if TOTAL_WRITES / b <= PER_KEY_LIMIT)
print("chosen fan-out:", chosen)

assert fan_out[4] == 16_000.0
assert fan_out[8] == 8_000.0
assert fan_out[16] == 4_000.0
assert chosen == 8
print("ok")


### Try it 8. Plan a backup schedule from a recovery point objective

**Task.** A table receives 2,000 writes per second. The recovery point objective allows losing at most 15 minutes of data, and a backup takes 4 minutes. Decide whether a 10-minute backup interval meets the objective, and compute the worst-case data loss.

**Hint.** Worst-case loss is `(interval + backup_duration)` minutes of writes, because a crash during a backup can lose the interval plus the in-progress backup window. Compare that to the 15-minute objective in seconds.

**Expected result.** Worst-case loss is 14 minutes (`1,680,000` writes), a 10-minute interval meets the objective, and a 15-minute interval does not.

In [ ]:
WRITES_PER_SECOND = 2_000
RPO_MINUTES = 15
BACKUP_MINUTES = 4


def worst_case_loss_minutes(interval_minutes):
    return interval_minutes + BACKUP_MINUTES


loss_10 = worst_case_loss_minutes(10)
loss_15 = worst_case_loss_minutes(15)
loss_10_writes = loss_10 * 60 * WRITES_PER_SECOND

print(f"10-minute interval -> {loss_10} minutes, {loss_10_writes:,} writes worst case")
print(f"15-minute interval -> {loss_15} minutes worst case")
print("10-minute interval meets RPO:", loss_10 <= RPO_MINUTES)

assert loss_10 == 14
assert loss_10_writes == 1_680_000
assert loss_10 <= RPO_MINUTES
assert loss_15 > RPO_MINUTES
print("ok")
